# DRIFT: Domain-Residual Rank Allocation for Parameter-Efficient Adaptation

Reproduces every experiment in the paper. Works on **Kaggle** and **Google Colab**.

**Kaggle:** in the right-hand panel set **Accelerator = GPU T4 x2** and **Internet = On**, then use *Save Version -> Save & Run All (Commit)* so it runs in the background. Each session stops starting new runs after `DEADLINE_HOURS`, so it always finishes inside Kaggle's 12-hour limit and saves `drift_results.zip`. To continue in a later session, upload that zip as a Kaggle Dataset (or a new version of it), attach it with *Add Input*, and run again: finished runs are restored and skipped.

**Colab:** *Runtime -> Change runtime type -> T4 GPU*, then *Run all*. Results are written straight to Google Drive (`MyDrive/drift_results/`), so a disconnect loses at most the run in progress: just *Run all* again.


In [ ]:
import os, sys, subprocess, time
SESSION_START = time.time()
ON_KAGGLE = os.path.exists('/kaggle/working')
ON_COLAB = 'google.colab' in sys.modules
WORK = '/kaggle/working' if ON_KAGGLE else '/content/drift'
os.makedirs(WORK, exist_ok=True)
os.chdir(WORK)
print('platform:', 'kaggle' if ON_KAGGLE else 'colab' if ON_COLAB else 'other',
      '| work dir:', WORK)
print(subprocess.run(['nvidia-smi','--query-gpu=name,memory.total',
                      '--format=csv'], capture_output=True, text=True).stdout)
import torch
NGPU = torch.cuda.device_count()
print('torch', torch.__version__, '| GPUs:', NGPU)
assert NGPU > 0, 'No GPU: enable a GPU accelerator/runtime first.'


The harness implements every PEFT method itself, so the only requirements are torch, transformers, pandas/pyarrow, scikit-learn and scipy -- all preinstalled on Kaggle and Colab. We only install what is genuinely missing, rather than upgrading packages and risking the environment.

In [ ]:
import importlib
need = []
for mod, pkg in [('torch','torch'), ('transformers','transformers'),
                 ('pandas','pandas'), ('pyarrow','pyarrow'),
                 ('sklearn','scikit-learn'), ('scipy','scipy')]:
    try:
        importlib.import_module(mod)
    except ImportError:
        need.append(pkg)
print('missing:', need or 'nothing')
if need:
    subprocess.run([sys.executable,'-m','pip','install','-q',*need], check=True)
import transformers
print('transformers', transformers.__version__)


## Write the source tree

In [ ]:
import base64, json
os.makedirs(os.path.join(WORK, 'src'), exist_ok=True)
PAYLOAD = json.loads(r'''{"common.py": "IiIiU2hhcmVkIHV0aWxpdGllczogc2VlZGluZywgbWV0cmljcywgcGFyYW1ldGVyIGFjY291bnRpbmcsIHRpbWluZy4iIiIKaW1wb3J0IGpzb24sIG9zLCByYW5kb20sIHRpbWUsIGhhc2hsaWIKaW1wb3J0IG51bXB5IGFzIG5wCgpkZWYgc2V0X3NlZWQoc2VlZDogaW50KToKICAgIHJhbmRvbS5zZWVkKHNlZWQpOyBucC5yYW5kb20uc2VlZChzZWVkKQogICAgdHJ5OgogICAgICAgIGltcG9ydCB0b3JjaAogICAgICAgIHRvcmNoLm1hbnVhbF9zZWVkKHNlZWQpOyB0b3JjaC5jdWRhLm1hbnVhbF9zZWVkX2FsbChzZWVkKQogICAgZXhjZXB0IEltcG9ydEVycm9yOgogICAgICAgIHBhc3MKCmRlZiBnZXRfZGV2aWNlKCk6CiAgICBpbXBvcnQgdG9yY2gKICAgIHJldHVybiB0b3JjaC5kZXZpY2UoImN1ZGEiIGlmIHRvcmNoLmN1ZGEuaXNfYXZhaWxhYmxlKCkgZWxzZSAiY3B1IikKCiMgLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLSBtZXRyaWNzCmRlZiBjbGZfbWV0cmljcyh5X3RydWUsIHlfcHJlZCwgbl9jbGFzc2VzKToKICAgICIiIk1pY3JvLUYxICg9PSBhY2N1cmFjeSBmb3Igc2luZ2xlLWxhYmVsKSBhbmQgbWFjcm8tRjEuIiIiCiAgICBmcm9tIHNrbGVhcm4ubWV0cmljcyBpbXBvcnQgZjFfc2NvcmUsIGFjY3VyYWN5X3Njb3JlCiAgICByZXR1cm4gewogICAgICAgICJtaWNyb19mMSI6IGZsb2F0KGYxX3Njb3JlKHlfdHJ1ZSwgeV9wcmVkLCBhdmVyYWdlPSJtaWNybyIsIHplcm9fZGl2aXNpb249MCkpLAogICAgICAgICJtYWNyb19mMSI6IGZsb2F0KGYxX3Njb3JlKHlfdHJ1ZSwgeV9wcmVkLCBhdmVyYWdlPSJtYWNybyIsIHplcm9fZGl2aXNpb249MCkpLAogICAgICAgICJhY2N1cmFjeSI6IGZsb2F0KGFjY3VyYWN5X3Njb3JlKHlfdHJ1ZSwgeV9wcmVkKSksCiAgICB9CgpkZWYgbXVsdGlsYWJlbF9tZXRyaWNzKHlfdHJ1ZSwgeV9wcm9iLCB0aHJlc2g9MC41KToKICAgICIiIkV4YW1wbGUtYmFzZWQgRjEgKEJMVVJCIGNvbnZlbnRpb24gZm9yIEhvQykgKyBtaWNyby9tYWNybyBGMS4iIiIKICAgIGZyb20gc2tsZWFybi5tZXRyaWNzIGltcG9ydCBmMV9zY29yZQogICAgeV9wcmVkID0gKHlfcHJvYiA+PSB0aHJlc2gpLmFzdHlwZShpbnQpCiAgICBpbnRlciA9ICh5X3ByZWQgKiB5X3RydWUpLnN1bSgxKQogICAgZGVub20gPSB5X3ByZWQuc3VtKDEpICsgeV90cnVlLnN1bSgxKQogICAgZXhfZjEgPSBucC53aGVyZShkZW5vbSA+IDAsIDIuMCAqIGludGVyIC8gbnAubWF4aW11bShkZW5vbSwgMWUtOSksIDEuMCkKICAgIHJldHVybiB7CiAgICAgICAgImV4YW1wbGVfZjEiOiBmbG9hdChleF9mMS5tZWFuKCkpLAogICAgICAgICJtaWNyb19mMSI6IGZsb2F0KGYxX3Njb3JlKHlfdHJ1ZSwgeV9wcmVkLCBhdmVyYWdlPSJtaWNybyIsIHplcm9fZGl2aXNpb249MCkpLAogICAgICAgICJtYWNyb19mMSI6IGZsb2F0KGYxX3Njb3JlKHlfdHJ1ZSwgeV9wcmVkLCBhdmVyYWdlPSJtYWNybyIsIHplcm9fZGl2aXNpb249MCkpLAogICAgfQoKIyAtLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tIHBhcmFtIGNvdW50aW5nCmRlZiBjb3VudF9wYXJhbXMobW9kZWwpOgogICAgdG90YWwgPSBzdW0ocC5udW1lbCgpIGZvciBwIGluIG1vZGVsLnBhcmFtZXRlcnMoKSkKICAgIHRyYWluYWJsZSA9IHN1bShwLm51bWVsKCkgZm9yIHAgaW4gbW9kZWwucGFyYW1ldGVycygpIGlmIHAucmVxdWlyZXNfZ3JhZCkKICAgIHJldHVybiB0b3RhbCwgdHJhaW5hYmxlCgpkZWYgY291bnRfYWRhcHRlcl9wYXJhbXMobW9kZWwpOgogICAgIiIiVHJhaW5hYmxlIHBhcmFtcyBleGNsdWRpbmcgdGhlIHRhc2sgaGVhZCAoaGVhZCBpcyByZXF1aXJlZCBieSBldmVyeSBtZXRob2QsCiAgICBzbyBidWRnZXQgY29tcGFyaXNvbnMgYXJlIG1hZGUgb24gdGhlICphZGFwdGVyKiBwYXJhbWV0ZXJzIG9ubHkpLiIiIgogICAgbiA9IDAKICAgIGZvciBuYW1lLCBwIGluIG1vZGVsLm5hbWVkX3BhcmFtZXRlcnMoKToKICAgICAgICBpZiBwLnJlcXVpcmVzX2dyYWQgYW5kIG5vdCBuYW1lLnN0YXJ0c3dpdGgoImhlYWQuIik6CiAgICAgICAgICAgIG4gKz0gcC5udW1lbCgpCiAgICByZXR1cm4gbgoKIyAtLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tIGlvCmRlZiBzYXZlX2pzb24ob2JqLCBwYXRoKToKICAgIG9zLm1ha2VkaXJzKG9zLnBhdGguZGlybmFtZShwYXRoKSwgZXhpc3Rfb2s9VHJ1ZSkKICAgIHdpdGggb3BlbihwYXRoLCAidyIsIGVuY29kaW5nPSJ1dGYtOCIpIGFzIGY6CiAgICAgICAganNvbi5kdW1wKG9iaiwgZiwgaW5kZW50PTIsIGRlZmF1bHQ9c3RyKQoKZGVmIGxvYWRfanNvbihwYXRoKToKICAgIHdpdGggb3BlbihwYXRoLCBlbmNvZGluZz0idXRmLTgiKSBhcyBmOgogICAgICAgIHJldHVybiBqc29uLmxvYWQoZikKCmNsYXNzIFRpbWVyOgogICAgZGVmIF9fZW50ZXJfXyhzZWxmKTogc2VsZi50MCA9IHRpbWUucGVyZl9jb3VudGVyKCk7IHJldHVybiBzZWxmCiAgICBkZWYgX19leGl0X18oc2VsZiwgKmEpOiBzZWxmLmR0ID0gdGltZS5wZXJmX2NvdW50ZXIoKSAtIHNlbGYudDAK", "data.py": "IiIiRGF0YXNldCBsb2FkaW5nIGZvciBiaW9tZWRpY2FsIGNsYXNzaWZpY2F0aW9uIHRhc2tzICsgZ2VuZXJhbC1kb21haW4gcmVmZXJlbmNlIGNvcnB1cy4KClRhc2tzCi0tLS0tCmNoZW1wcm90IDogMTMtd2F5IGNoZW1pY2FsLXByb3RlaW4gcmVsYXRpb24gY2xhc3NpZmljYXRpb24sIHNlbnRlbmNlIGxldmVsLCBQdWJNZWQKICAgICAgICAgICBhYnN0cmFjdHMgd2l0aCBlbnRpdHkgbWVudGlvbnMgbWFya2VkIGJ5IDw8ID4+IGFuZCBbWyBdXS4gIENhbm9uaWNhbAogICAgICAgICAgIERBUFQvVEFQVCBhbmQgQkxVUkIgdGFzay4gIDQxNjkgLyAyNDI3IC8gMzQ2OS4KcmN0MjBrICAgOiA1LXdheSByaGV0b3JpY2FsLXJvbGUgY2xhc3NpZmljYXRpb24gb2Ygc2VudGVuY2VzIGluIFJDVCBhYnN0cmFjdHMKICAgICAgICAgICAoQkFDS0dST1VORCAvIE9CSkVDVElWRSAvIE1FVEhPRFMgLyBSRVNVTFRTIC8gQ09OQ0xVU0lPTlMpLgpob2MgICAgICA6IEhhbGxtYXJrcyBvZiBDYW5jZXIgLS0gMTAtbGFiZWwgbXVsdGktbGFiZWwgY2xhc3NpZmljYXRpb24gb2YgUHViTWVkCiAgICAgICAgICAgYWJzdHJhY3RzLiAgUmVjb25zdHJ1Y3RlZCBhdCBkb2N1bWVudCBsZXZlbCAodGhlIEJMVVJCIGZvcm11bGF0aW9uKSBieQogICAgICAgICAgIGdyb3VwaW5nIHRoZSBzZW50ZW5jZS1sZXZlbCByZWxlYXNlIG9uIFBNSUQgYW5kIHRha2luZyB0aGUgdW5pb24gb2YKICAgICAgICAgICBoYWxsbWFyayBsYWJlbHM7IHRoZSAibm8gaGFsbG1hcmsiIGNsYXNzIGlzIGRyb3BwZWQsIHNvIGFic3RyYWN0cyB3aXRoCiAgICAgICAgICAgbm8gaGFsbG1hcmsgY2FycnkgYW4gYWxsLXplcm8gdGFyZ2V0LgoKUmVmZXJlbmNlIGNvcnB1cwotLS0tLS0tLS0tLS0tLS0tCndpa2l0ZXh0LTEwMyAocmF3KSAtLSBhIGdlbmVyYWwtZG9tYWluIHByb3h5IGZvciB0aGUgcHJldHJhaW5pbmcgZGlzdHJpYnV0aW9uLCB1c2VkCmJ5IERSSUZUIHRvIGVzdGltYXRlIHRoZSBzdWJzcGFjZSB0aGUgYmFzZSBtb2RlbCBoYXMgYWxyZWFkeSBiZWVuIG9wdGltaXNlZCBmb3IuClRocmVlIGNvbnRyb2xzIHJlcGxhY2UgaXQgKGxvYWRfcmVmZXJlbmNlX2NvcnB1cyhraW5kPS4uLikpOiBDTk4vRGFpbHlNYWlsIG5ld3MKYXJ0aWNsZXMgKGEgc2Vjb25kIGdlbmVyYWwtZG9tYWluIGNvcnB1cyksIFdpa2lUZXh0IHdpdGggdGhlIHdvcmQgb3JkZXIgc2h1ZmZsZWQKaW5zaWRlIGVhY2ggcGFzc2FnZSwgYW5kIHVuaWZvcm1seSByYW5kb20gdm9jYWJ1bGFyeSB0b2tlbnMuCiIiIgppbXBvcnQganNvbgppbXBvcnQgb3MKaW1wb3J0IHJhbmRvbQppbXBvcnQgcmUKCkRBVEEgPSBvcy5wYXRoLmpvaW4ob3MucGF0aC5kaXJuYW1lKG9zLnBhdGguZGlybmFtZShvcy5wYXRoLmFic3BhdGgoX19maWxlX18pKSksICJkYXRhIikKCgpkZWYgX3JlYWRfanNvbmwocGF0aCk6CiAgICByb3dzID0gW10KICAgIHdpdGggb3BlbihwYXRoLCBlbmNvZGluZz0idXRmLTgiKSBhcyBmOgogICAgICAgIGZvciBsaW5lIGluIGY6CiAgICAgICAgICAgIGxpbmUgPSBsaW5lLnN0cmlwKCkKICAgICAgICAgICAgaWYgbGluZToKICAgICAgICAgICAgICAgIHJvd3MuYXBwZW5kKGpzb24ubG9hZHMobGluZSkpCiAgICByZXR1cm4gcm93cwoKCmRlZiBfc3Vic2FtcGxlKHRleHRzLCBsYWJlbHMsIG4sIHNlZWQ9MCk6CiAgICBpZiBuIGlzIE5vbmUgb3IgbiA+PSBsZW4odGV4dHMpOgogICAgICAgIHJldHVybiB0ZXh0cywgbGFiZWxzCiAgICBybmcgPSByYW5kb20uUmFuZG9tKHNlZWQpCiAgICBpZHggPSBsaXN0KHJhbmdlKGxlbih0ZXh0cykpKQogICAgcm5nLnNodWZmbGUoaWR4KQogICAgaWR4ID0gc29ydGVkKGlkeFs6bl0pCiAgICByZXR1cm4gW3RleHRzW2ldIGZvciBpIGluIGlkeF0sIFtsYWJlbHNbaV0gZm9yIGkgaW4gaWR4XQoKCmRlZiBfanNvbmxfdGFzayhmb2xkZXIsIG1heF90cmFpbj1Ob25lLCBzZWVkPTAsIGV2YWxfY2FwPU5vbmUsIG5hbWU9IiIpOgogICAgcmF3LCBsYWJlbF9zZXQgPSB7fSwgTm9uZQogICAgZm9yIHNwbGl0LCBmbiBpbiBbKCJ0cmFpbiIsICJ0cmFpbi5qc29ubCIpLCAoImRldiIsICJkZXYuanNvbmwiKSwgKCJ0ZXN0IiwgInRlc3QuanNvbmwiKV06CiAgICAgICAgcm93cyA9IF9yZWFkX2pzb25sKG9zLnBhdGguam9pbihEQVRBLCBmb2xkZXIsIGZuKSkKICAgICAgICByYXdbc3BsaXRdID0gKFtyWyJ0ZXh0Il0gZm9yIHIgaW4gcm93c10sIFtyWyJsYWJlbCJdIGZvciByIGluIHJvd3NdKQogICAgICAgIGlmIGxhYmVsX3NldCBpcyBOb25lOgogICAgICAgICAgICBsYWJlbF9zZXQgPSBzb3J0ZWQoe3JbImxhYmVsIl0gZm9yIHIgaW4gcm93c30pCiAgICBsMmkgPSB7bDogaSBmb3IgaSwgbCBpbiBlbnVtZXJhdGUobGFiZWxfc2V0KX0KICAgIG91dCA9IHt9CiAgICBmb3Igc3BsaXQsICh0LCBsKSBpbiByYXcuaXRlbXMoKToKICAgICAgICB5ID0gW2wyaVt4XSBmb3IgeCBpbiBsXQogICAgICAgIGlmIHNwbGl0ID09ICJ0cmFpbiI6CiAgICAgICAgICAgIHQsIHkgPSBfc3Vic2FtcGxlKHQsIHksIG1heF90cmFpbiwgc2VlZCkKICAgICAgICBlbGlmIGV2YWxfY2FwIGlzIG5vdCBOb25lOgogICAgICAgICAgICAjIENhcHBlZCB3aXRoIGEgRklYRUQgc2VlZCBzbyBldmVyeSBtZXRob2Qvc2VlZCBzZWVzIHRoZSBpZGVudGljYWwKICAgICAgICAgICAgIyBldmFsdWF0aW9uIHN1YnNldDsgY29tcGFyaXNvbnMgdGhlcmVmb3JlIHN0YXkgcGFpcmVkLgogICAgICAgICAgICB0LCB5ID0gX3N1YnNhbXBsZSh0LCB5LCBldmFsX2NhcCwgMTIzNDUpCiAgICAgICAgb3V0W3NwbGl0XSA9ICh0LCB5KQogICAgcmV0dXJuIHsic3BsaXRzIjogb3V0LCAibnVtX2xhYmVscyI6IGxlbihsYWJlbF9zZXQpLCAibGFiZWxzIjogbGFiZWxfc2V0LAogICAgICAgICAgICAibXVsdGlsYWJlbCI6IEZhbHNlLCAibWV0cmljIjogIm1pY3JvX2YxIiwgIm5hbWUiOiBuYW1lfQoKCmRlZiBsb2FkX2NoZW1wcm90KG1heF90cmFpbj1Ob25lLCBzZWVkPTApOgogICAgcmV0dXJuIF9qc29ubF90YXNrKCJjaGVtcHJvdCIsIG1heF90cmFpbiwgc2VlZCwgZXZhbF9jYXA9Tm9uZSwgbmFtZT0iY2hlbXByb3QiKQoKCmRlZiBsb2FkX3JjdDIwayhtYXhfdHJhaW49NTAwMCwgc2VlZD0wKToKICAgIHJldHVybiBfanNvbmxfdGFzaygicmN0MjBrIiwgbWF4X3RyYWluLCBzZWVkLCBldmFsX2NhcD02MDAwLCBuYW1lPSJyY3QyMGsiKQoKCmRlZiBsb2FkX2hvYyhtYXhfdHJhaW49Tm9uZSwgc2VlZD0wKToKICAgIGltcG9ydCBwYW5kYXMgYXMgcGQKICAgIGltcG9ydCBudW1weSBhcyBucAogICAgTk9ORV9DTEFTUyA9IDcKICAgIGZyYW1lcyA9IHt9CiAgICBmb3Igc3BsaXQsIGZuIGluIFsoInRyYWluIiwgInRyYWluLnBhcnF1ZXQiKSwgKCJkZXYiLCAidmFsaWRhdGlvbi5wYXJxdWV0IiksCiAgICAgICAgICAgICAgICAgICAgICAoInRlc3QiLCAidGVzdC5wYXJxdWV0IildOgogICAgICAgIGRmID0gcGQucmVhZF9wYXJxdWV0KG9zLnBhdGguam9pbihEQVRBLCAiaG9jIiwgZm4pKQogICAgICAgIGRmWyJwbWlkIl0gPSBkZlsiZG9jdW1lbnRfaWQiXS5zdHIuc3BsaXQoIl8iKS5zdHJbMF0KICAgICAgICBkZlsic2lkeCJdID0gZGZbImRvY3VtZW50X2lkIl0uc3RyLnNwbGl0KCJfIikuc3RyWzFdLmFzdHlwZShpbnQpCiAgICAgICAgZyA9IGRmLnNvcnRfdmFsdWVzKFsicG1pZCIsICJzaWR4Il0pLmdyb3VwYnkoInBtaWQiKQogICAgICAgIHRleHRzID0gZ1sidGV4dCJdLmFwcGx5KGxhbWJkYSBzOiAiICIuam9pbihzKSkKICAgICAgICBsYWJzID0gZ1sibGFiZWwiXS5hcHBseSgKICAgICAgICAgICAgbGFtYmRhIHM6IHNvcnRlZCh7aW50KHgpIGZvciBsIGluIHMgZm9yIHggaW4gbCBpZiBpbnQoeCkgIT0gTk9ORV9DTEFTU30pKQogICAgICAgIGZyYW1lc1tzcGxpdF0gPSAodGV4dHMudG9saXN0KCksIGxhYnMudG9saXN0KCkpCgogICAgcHJlc2VudCA9IHNvcnRlZCh7eCBmb3IgXywgbGFicyBpbiBmcmFtZXMudmFsdWVzKCkgZm9yIGwgaW4gbGFicyBmb3IgeCBpbiBsfSkKICAgIGwyaSA9IHtjOiBpIGZvciBpLCBjIGluIGVudW1lcmF0ZShwcmVzZW50KX0KICAgIG91dCA9IHt9CiAgICBmb3Igc3BsaXQsICh0LCBsYWJzKSBpbiBmcmFtZXMuaXRlbXMoKToKICAgICAgICB5ID0gbnAuemVyb3MoKGxlbihsYWJzKSwgbGVuKHByZXNlbnQpKSwgZHR5cGU9ImZsb2F0MzIiKQogICAgICAgIGZvciBpLCBsIGluIGVudW1lcmF0ZShsYWJzKToKICAgICAgICAgICAgZm9yIGMgaW4gbDoKICAgICAgICAgICAgICAgIHlbaSwgbDJpW2NdXSA9IDEuMAogICAgICAgIHkgPSBbcm93IGZvciByb3cgaW4geV0KICAgICAgICBpZiBzcGxpdCA9PSAidHJhaW4iOgogICAgICAgICAgICB0LCB5ID0gX3N1YnNhbXBsZSh0LCB5LCBtYXhfdHJhaW4sIHNlZWQpCiAgICAgICAgb3V0W3NwbGl0XSA9ICh0LCB5KQogICAgcmV0dXJuIHsic3BsaXRzIjogb3V0LCAibnVtX2xhYmVscyI6IGxlbihwcmVzZW50KSwgImxhYmVscyI6IHByZXNlbnQsCiAgICAgICAgICAgICJtdWx0aWxhYmVsIjogVHJ1ZSwgIm1ldHJpYyI6ICJleGFtcGxlX2YxIiwgIm5hbWUiOiAiaG9jIn0KCgojIE1UU2FtcGxlcyBsYWJlbHMgdGhhdCBuYW1lIGEgZG9jdW1lbnQgdHlwZSBvciBhIGNhdGNoLWFsbCByYXRoZXIgdGhhbiBhCiMgbWVkaWNhbCBzcGVjaWFsdHk7IHRoZWlyIG5vdGVzIGFyZSBkdXBsaWNhdGVkIHVuZGVyIHNwZWNpZmljIHNwZWNpYWx0aWVzLgpNVFNfRFJPUCA9IHsiU3VyZ2VyeSIsICJDb25zdWx0IC0gSGlzdG9yeSBhbmQgUGh5LiIsICJTT0FQIC8gQ2hhcnQgLyBQcm9ncmVzcyBOb3RlcyIsCiAgICAgICAgICAgICJEaXNjaGFyZ2UgU3VtbWFyeSIsICJFbWVyZ2VuY3kgUm9vbSBSZXBvcnRzIiwgIk9mZmljZSBOb3RlcyIsICJMZXR0ZXJzIiwKICAgICAgICAgICAgIklNRS1RTUUtV29yayBDb21wIGV0Yy4iLCAiR2VuZXJhbCBNZWRpY2luZSJ9Ck1UU19NSU4gPSA3MCAgICAgICAgICAjIGtlZXAgc3BlY2lhbHRpZXMgd2l0aCBhdCBsZWFzdCB0aGlzIG1hbnkgdW5hbWJpZ3VvdXMgbm90ZXMKIyB0aGUgcmVsZWFzZSdzIENsYXNzTGFiZWwgb3JkZXIgKGl0cyBuYW1lcyBjYXJyeSBhIGxlYWRpbmcgc3BhY2UsIHN0cmlwcGVkIGhlcmUpCk1UU19OQU1FUyA9IFsKICAgICJQYWluIE1hbmFnZW1lbnQiLCAiQ2hpcm9wcmFjdGljIiwgIlBvZGlhdHJ5IiwgIlBlZGlhdHJpY3MgLSBOZW9uYXRhbCIsCiAgICAiRGlzY2hhcmdlIFN1bW1hcnkiLCAiQ29zbWV0aWMgLyBQbGFzdGljIFN1cmdlcnkiLCAiTmV1cm9sb2d5IiwgIkVuZG9jcmlub2xvZ3kiLAogICAgIlJoZXVtYXRvbG9neSIsICJPcnRob3BlZGljIiwgIkRlbnRpc3RyeSIsICJBbGxlcmd5IC8gSW1tdW5vbG9neSIsCiAgICAiUHN5Y2hpYXRyeSAvIFBzeWNob2xvZ3kiLCAiQ29uc3VsdCAtIEhpc3RvcnkgYW5kIFBoeS4iLCAiRGVybWF0b2xvZ3kiLAogICAgIlJhZGlvbG9neSIsICJTcGVlY2ggLSBMYW5ndWFnZSIsICJQaHlzaWNhbCBNZWRpY2luZSAtIFJlaGFiIiwgIlNsZWVwIE1lZGljaW5lIiwKICAgICJIb3NwaWNlIC0gUGFsbGlhdGl2ZSBDYXJlIiwgIkRpZXRzIGFuZCBOdXRyaXRpb25zIiwgIlVyb2xvZ3kiLAogICAgIkVOVCAtIE90b2xhcnluZ29sb2d5IiwgIkdhc3Ryb2VudGVyb2xvZ3kiLCAiTGV0dGVycyIsICJTdXJnZXJ5IiwgIkJhcmlhdHJpY3MiLAogICAgIk9waHRoYWxtb2xvZ3kiLCAiTmV1cm9zdXJnZXJ5IiwgIkVtZXJnZW5jeSBSb29tIFJlcG9ydHMiLCAiTmVwaHJvbG9neSIsCiAgICAiTGFiIE1lZGljaW5lIC0gUGF0aG9sb2d5IiwgIk9mZmljZSBOb3RlcyIsICJDYXJkaW92YXNjdWxhciAvIFB1bG1vbmFyeSIsCiAgICAiU09BUCAvIENoYXJ0IC8gUHJvZ3Jlc3MgTm90ZXMiLCAiQXV0b3BzeSIsICJHZW5lcmFsIE1lZGljaW5lIiwKICAgICJJTUUtUU1FLVdvcmsgQ29tcCBldGMuIiwgIk9ic3RldHJpY3MgLyBHeW5lY29sb2d5IiwgIkhlbWF0b2xvZ3kgLSBPbmNvbG9neSJdCgoKZGVmIGxvYWRfbXRzYW1wbGVzKG1heF90cmFpbj1Ob25lLCBzZWVkPTApOgogICAgIiIiQ2xpbmljYWwtc3R5bGUgc3BlY2lhbHR5IGNsYXNzaWZpY2F0aW9uIGZyb20gTVRTYW1wbGVzIHRyYW5zY3JpcHRpb25zLgoKICAgIFRoZSBwdWJsaWMgcmVsZWFzZSAoZ2FsaWxlby1haS9tZWRpY2FsX3RyYW5zY3JpcHRpb25fNDAsIDQsNTAwICsgNTAwIG5vdGVzLAogICAgNDAgbGFiZWxzKSBtaXhlcyBzcGVjaWFsdGllcyB3aXRoIGRvY3VtZW50IHR5cGVzIGFuZCBsaXN0cyBtYW55IG5vdGVzIHVuZGVyCiAgICBzZXZlcmFsIGxhYmVscy4gV2UgcG9vbCBpdHMgdHdvIHNwbGl0cywgZHJvcCB0aGUgZG9jdW1lbnQtdHlwZSBhbmQgY2F0Y2gtYWxsCiAgICBsYWJlbHMgKE1UU19EUk9QKSwgZHJvcCBldmVyeSBub3RlIHRoYXQgYXBwZWFycyB1bmRlciBtb3JlIHRoYW4gb25lIHJlbWFpbmluZwogICAgbGFiZWwsIGtlZXAgdGhlIHNwZWNpYWx0aWVzIHdpdGggYXQgbGVhc3QgTVRTX01JTiBub3RlcywgYW5kIHNwbGl0IGVhY2gKICAgIHNwZWNpYWx0eSA3MC8xNS8xNSBpbnRvIHRyYWluL2Rldi90ZXN0IHdpdGggYSBmaXhlZCBzZWVkLCBzbyBldmVyeSBtZXRob2Qgc2VlcwogICAgaWRlbnRpY2FsIGRhdGEuIFNjb3JlZCB3aXRoIG1pY3JvLUYxLiIiIgogICAgaW1wb3J0IHB5YXJyb3cucGFycXVldCBhcyBwcQogICAgcm93cyA9IFtdCiAgICBuYW1lcyA9IGxpc3QoTVRTX05BTUVTKQogICAgZm9yIGZuIGluICgidHJhaW4ucGFycXVldCIsICJ0ZXN0LnBhcnF1ZXQiKToKICAgICAgICBwID0gb3MucGF0aC5qb2luKERBVEEsICJtdHNhbXBsZXMiLCBmbikKICAgICAgICBtZXRhID0gcHEucmVhZF9zY2hlbWEocCkubWV0YWRhdGEgb3Ige30KICAgICAgICBpZiBiImh1Z2dpbmdmYWNlIiBpbiBtZXRhOgogICAgICAgICAgICAjIHRoZSBmaWxlJ3Mgb3duIGxhYmVsIG9yZGVyLCBpZiBpdCBjYXJyaWVzIG9uZSwgbXVzdCBhZ3JlZQogICAgICAgICAgICBmZWF0cyA9IGpzb24ubG9hZHMobWV0YVtiImh1Z2dpbmdmYWNlIl0pLmdldCgiaW5mbyIsIHt9KS5nZXQoImZlYXR1cmVzIiwge30pCiAgICAgICAgICAgIG93biA9IFtuLnN0cmlwKCkgZm9yIG4gaW4gZmVhdHMuZ2V0KCJsYWJlbCIsIHt9KS5nZXQoIm5hbWVzIiwgW10pXQogICAgICAgICAgICBpZiBvd24gYW5kIG93biAhPSBuYW1lczoKICAgICAgICAgICAgICAgIHJhaXNlIFZhbHVlRXJyb3IoIk1UU2FtcGxlcyBsYWJlbCBvcmRlciBkaWZmZXJzIGZyb20gTVRTX05BTUVTIikKICAgICAgICB0ID0gcHEucmVhZF90YWJsZShwKS50b19weWRpY3QoKQogICAgICAgIHJvd3MgKz0gWyh4LnN0cmlwKCksIGludCh5KSkgZm9yIHgsIHkgaW4gemlwKHRbInRleHQiXSwgdFsibGFiZWwiXSkKICAgICAgICAgICAgICAgICBpZiBpc2luc3RhbmNlKHgsIHN0cikgYW5kIHguc3RyaXAoKV0KICAgIGxhYmVsc19vZiA9IHt9CiAgICBmb3IgeCwgeSBpbiByb3dzOgogICAgICAgIGxhYmVsc19vZi5zZXRkZWZhdWx0KHgsIHNldCgpKS5hZGQobmFtZXNbeV0pCiAgICAjIGtlZXAgYSBub3RlIHdoZW4gZXhhY3RseSBvbmUgc3BlY2lhbHR5IHJlbWFpbnMgb25jZSB0aGUgZHJvcHBlZCBsYWJlbHMgYXJlCiAgICAjIHJlbW92ZWQ6IGEgbm90ZSBjcm9zcy1saXN0ZWQgdW5kZXIgIlN1cmdlcnkiIGFuZCAiT3J0aG9wZWRpYyIgaXMgYW4KICAgICMgb3J0aG9wZWRpYyBub3RlOyBvbmUgbGlzdGVkIHVuZGVyIHR3byBzcGVjaWFsdGllcyBpcyBhbWJpZ3VvdXMgYW5kIGRyb3BwZWQKICAgIGtlZXAgPSB7eDogbmV4dChpdGVyKGxzIC0gTVRTX0RST1ApKSBmb3IgeCwgbHMgaW4gbGFiZWxzX29mLml0ZW1zKCkKICAgICAgICAgICAgaWYgbGVuKGxzIC0gTVRTX0RST1ApID09IDF9CiAgICBieSA9IHt9CiAgICBmb3IgeCwgbGFiIGluIHNvcnRlZChrZWVwLml0ZW1zKCkpOgogICAgICAgIGJ5LnNldGRlZmF1bHQobGFiLCBbXSkuYXBwZW5kKHgpCiAgICBjbGFzc2VzID0gc29ydGVkKGxhYiBmb3IgbGFiLCB4cyBpbiBieS5pdGVtcygpIGlmIGxlbih4cykgPj0gTVRTX01JTikKICAgIHJuZyA9IHJhbmRvbS5SYW5kb20oc2VlZCArIDIwMjQpCiAgICBzcGxpdCA9IHsidHJhaW4iOiAoW10sIFtdKSwgImRldiI6IChbXSwgW10pLCAidGVzdCI6IChbXSwgW10pfQogICAgZm9yIGNpLCBsYWIgaW4gZW51bWVyYXRlKGNsYXNzZXMpOgogICAgICAgIHhzID0gbGlzdChieVtsYWJdKQogICAgICAgIHJuZy5zaHVmZmxlKHhzKQogICAgICAgIG5fZGV2ID0gbl90ZXN0ID0gbWF4KDEsIHJvdW5kKDAuMTUgKiBsZW4oeHMpKSkKICAgICAgICBwYXJ0cyA9IHsidGVzdCI6IHhzWzpuX3Rlc3RdLCAiZGV2IjogeHNbbl90ZXN0Om5fdGVzdCArIG5fZGV2XSwKICAgICAgICAgICAgICAgICAidHJhaW4iOiB4c1tuX3Rlc3QgKyBuX2RldjpdfQogICAgICAgIGZvciBzLCBwYXJ0IGluIHBhcnRzLml0ZW1zKCk6CiAgICAgICAgICAgIHNwbGl0W3NdWzBdLmV4dGVuZChwYXJ0KQogICAgICAgICAgICBzcGxpdFtzXVsxXS5leHRlbmQoW2NpXSAqIGxlbihwYXJ0KSkKICAgIG91dCA9IHt9CiAgICBmb3IgcywgKHQsIHkpIGluIHNwbGl0Lml0ZW1zKCk6CiAgICAgICAgb3JkZXIgPSBsaXN0KHJhbmdlKGxlbih0KSkpCiAgICAgICAgcmFuZG9tLlJhbmRvbShzZWVkICsgNykuc2h1ZmZsZShvcmRlcikKICAgICAgICB0LCB5ID0gW3RbaV0gZm9yIGkgaW4gb3JkZXJdLCBbeVtpXSBmb3IgaSBpbiBvcmRlcl0KICAgICAgICBpZiBzID09ICJ0cmFpbiI6CiAgICAgICAgICAgIHQsIHkgPSBfc3Vic2FtcGxlKHQsIHksIG1heF90cmFpbiwgc2VlZCkKICAgICAgICBvdXRbc10gPSAodCwgeSkKICAgIHJldHVybiB7InNwbGl0cyI6IG91dCwgIm51bV9sYWJlbHMiOiBsZW4oY2xhc3NlcyksICJsYWJlbHMiOiBjbGFzc2VzLAogICAgICAgICAgICAibXVsdGlsYWJlbCI6IEZhbHNlLCAibWV0cmljIjogIm1pY3JvX2YxIiwgIm5hbWUiOiAibXRzYW1wbGVzIn0KCgpkZWYgbG9hZF90YXNrKG5hbWUsIG1heF90cmFpbj1Ob25lLCBzZWVkPTApOgogICAgaWYgbmFtZSA9PSAiY2hlbXByb3QiOgogICAgICAgIHJldHVybiBsb2FkX2NoZW1wcm90KG1heF90cmFpbiwgc2VlZCkKICAgIGlmIG5hbWUgPT0gInJjdDIwayI6CiAgICAgICAgcmV0dXJuIGxvYWRfcmN0MjBrKG1heF90cmFpbiBpZiBtYXhfdHJhaW4gaXMgbm90IE5vbmUgZWxzZSA1MDAwLCBzZWVkKQogICAgaWYgbmFtZSA9PSAiaG9jIjoKICAgICAgICByZXR1cm4gbG9hZF9ob2MobWF4X3RyYWluLCBzZWVkKQogICAgaWYgbmFtZSA9PSAibXRzYW1wbGVzIjoKICAgICAgICByZXR1cm4gbG9hZF9tdHNhbXBsZXMobWF4X3RyYWluLCBzZWVkKQogICAgcmFpc2UgVmFsdWVFcnJvcigidW5rbm93biB0YXNrICIgKyBuYW1lKQoKClRBU0tfTUFYTEVOID0geyJjaGVtcHJvdCI6IDEyOCwgInJjdDIwayI6IDk2LCAiaG9jIjogNTEyLCAibXRzYW1wbGVzIjogNTEyfQoKClJFRkVSRU5DRV9LSU5EUyA9ICgid2lraXRleHQiLCAibmV3cyIsICJzaHVmZmxlZCIsICJyYW5kb20iKQoKCmRlZiBfY2xlYW5fbmV3cyh0KToKICAgICIiIlN0cmlwIHRoZSBDTk4vRGFpbHlNYWlsIGJ5bGluZXMgYW5kIHRpbWUgc3RhbXBzIHRoYXQgb3BlbiBtYW55IGFydGljbGVzLiIiIgogICAgaGVhZCA9IHRbOjQwMF0KICAgIGN1dCA9IDAKICAgIGZvciBwYXQgaW4gKHIiVVBEQVRFRDpccypcLlxzKlteLl0qXC5ccyoiLCByIlBVQkxJU0hFRDpccypcLlxzKlteLl0qXC5ccyoiLAogICAgICAgICAgICAgICAgciJMYXN0IHVwZGF0ZWQgYXRbXi5dKlwuXHMqIik6CiAgICAgICAgZm9yIG0gaW4gcmUuZmluZGl0ZXIocGF0LCBoZWFkKToKICAgICAgICAgICAgY3V0ID0gbWF4KGN1dCwgbS5lbmQoKSkKICAgIHQgPSB0W2N1dDpdCiAgICBtID0gcmUubWF0Y2gociJeLnswLDgwfT9cKENOTlwpXHMqLS1ccyoiLCB0KQogICAgaWYgbToKICAgICAgICB0ID0gdFttLmVuZCgpOl0KICAgIHJldHVybiByZS5zdWIociJccytcLlxzKyIsICIuICIsIHQpLnN0cmlwKCkKCgpkZWYgbG9hZF9yZWZlcmVuY2VfY29ycHVzKG5fZG9jcz0yMDAwLCBtaW5fY2hhcnM9MjAwLCBzZWVkPTAsIGtpbmQ9Indpa2l0ZXh0IiwKICAgICAgICAgICAgICAgICAgICAgICAgICB0b2tlbml6ZXI9Tm9uZSwgbl90b2tlbnM9MTI2KToKICAgICIiIkdlbmVyYWwtZG9tYWluIHJlZmVyZW5jZSB0ZXh0LgoKICAgIGtpbmQ9Indpa2l0ZXh0IiA6IFdpa2lUZXh0LTEwMyBwYXJhZ3JhcGhzICh2YWxpZGF0aW9uICsgdGVzdCksIHRoZSBkZWZhdWx0LgogICAga2luZD0ibmV3cyIgICAgIDogQ05OL0RhaWx5TWFpbCBuZXdzIGFydGljbGVzICh0ZXN0IHNwbGl0KSwgYSBzZWNvbmQKICAgICAgICAgICAgICAgICAgICAgIGdlbmVyYWwtZG9tYWluIGNvcnB1cy4KICAgIGtpbmQ9InNodWZmbGVkIiA6IHRoZSBXaWtpVGV4dCBwYXNzYWdlcyB3aXRoIHRoZWlyIHdvcmQgb3JkZXIgc2h1ZmZsZWQgaW5zaWRlCiAgICAgICAgICAgICAgICAgICAgICBlYWNoIHBhc3NhZ2UgKHNhbWUgd29yZHMsIG5vIHN5bnRheCkuCiAgICBraW5kPSJyYW5kb20iICAgOiBsaXN0cyBvZiBuX3Rva2VucyB0b2tlbiBpZHMgZHJhd24gdW5pZm9ybWx5IGZyb20gdGhlCiAgICAgICAgICAgICAgICAgICAgICB0b2tlbml6ZXIncyB2b2NhYnVsYXJ5IChzcGVjaWFsIHRva2VucyBleGNsdWRlZCk7IHRoZXNlIGFyZQogICAgICAgICAgICAgICAgICAgICAgZmVkIHRvIHRoZSBtb2RlbCBhcyBpZHMsIG5ldmVyIHJlLXRva2VuaXNlZC4KICAgICIiIgogICAgaW1wb3J0IHBhbmRhcyBhcyBwZAogICAgaWYga2luZCA9PSAicmFuZG9tIjoKICAgICAgICBpbXBvcnQgbnVtcHkgYXMgbnAKICAgICAgICBybmcgPSBucC5yYW5kb20uZGVmYXVsdF9ybmcoc2VlZCkKICAgICAgICBzcGVjaWFsID0gc2V0KHRva2VuaXplci5hbGxfc3BlY2lhbF9pZHMpCiAgICAgICAgdm9jYWIgPSBucC5hcnJheShbaSBmb3IgaSBpbiByYW5nZSh0b2tlbml6ZXIudm9jYWJfc2l6ZSkgaWYgaSBub3QgaW4gc3BlY2lhbF0pCiAgICAgICAgcmV0dXJuIFt2b2NhYltybmcuaW50ZWdlcnMoMCwgbGVuKHZvY2FiKSwgc2l6ZT1uX3Rva2VucyldLnRvbGlzdCgpCiAgICAgICAgICAgICAgICBmb3IgXyBpbiByYW5nZShuX2RvY3MpXQogICAgaWYga2luZCA9PSAibmV3cyI6CiAgICAgICAgZGYgPSBwZC5yZWFkX3BhcnF1ZXQob3MucGF0aC5qb2luKERBVEEsICJyZWZlcmVuY2UiLCAiY25uX2RhaWx5bWFpbF90ZXN0LnBhcnF1ZXQiKSkKICAgICAgICB0ZXh0cyA9IFtfY2xlYW5fbmV3cyh0KSBmb3IgdCBpbiBkZlsiYXJ0aWNsZSJdLnRvbGlzdCgpIGlmIGlzaW5zdGFuY2UodCwgc3RyKV0KICAgICAgICB0ZXh0cyA9IFt0IGZvciB0IGluIHRleHRzIGlmIGxlbih0KSA+PSBtaW5fY2hhcnNdCiAgICBlbHNlOgogICAgICAgIGRmcyA9IFtdCiAgICAgICAgZm9yIGZuIGluICgid2lraXRleHRfdmFsLnBhcnF1ZXQiLCAid2lraXRleHRfdGVzdC5wYXJxdWV0Iik6CiAgICAgICAgICAgIHAgPSBvcy5wYXRoLmpvaW4oREFUQSwgInJlZmVyZW5jZSIsIGZuKQogICAgICAgICAgICBpZiBvcy5wYXRoLmV4aXN0cyhwKToKICAgICAgICAgICAgICAgIGRmcy5hcHBlbmQocGQucmVhZF9wYXJxdWV0KHApKQogICAgICAgIGRmID0gcGQuY29uY2F0KGRmcywgaWdub3JlX2luZGV4PVRydWUpCiAgICAgICAgdGV4dHMgPSBbdC5zdHJpcCgpIGZvciB0IGluIGRmWyJ0ZXh0Il0udG9saXN0KCkgaWYgaXNpbnN0YW5jZSh0LCBzdHIpXQogICAgICAgIHRleHRzID0gW3QgZm9yIHQgaW4gdGV4dHMgaWYgbGVuKHQpID49IG1pbl9jaGFycyBhbmQgbm90IHQuc3RhcnRzd2l0aCgiPSIpXQogICAgcm5nID0gcmFuZG9tLlJhbmRvbShzZWVkKQogICAgcm5nLnNodWZmbGUodGV4dHMpCiAgICB0ZXh0cyA9IHRleHRzWzpuX2RvY3NdCiAgICBpZiBraW5kID09ICJzaHVmZmxlZCI6CiAgICAgICAgc3JuZyA9IHJhbmRvbS5SYW5kb20oc2VlZCArIDEpCiAgICAgICAgb3V0ID0gW10KICAgICAgICBmb3IgdCBpbiB0ZXh0czoKICAgICAgICAgICAgd29yZHMgPSB0LnNwbGl0KCkKICAgICAgICAgICAgc3JuZy5zaHVmZmxlKHdvcmRzKQogICAgICAgICAgICBvdXQuYXBwZW5kKCIgIi5qb2luKHdvcmRzKSkKICAgICAgICB0ZXh0cyA9IG91dAogICAgcmV0dXJuIHRleHRzCg==", "drift.py": "...", "peft_methods.py": "IiIiVW5pZmllZCBpbXBsZW1lbnRhdGlvbnMgb2YgdGhlIFBFRlQgbWV0aG9kcyBjb21wYXJlZCBpbiB0aGUgcGFwZXIuCgpFdmVyeXRoaW5nIGlzIGltcGxlbWVudGVkIGluc2lkZSBvbmUgZnJhbWV3b3JrIHNvIHRoYXQgdHJhaW5hYmxlLXBhcmFtZXRlciBidWRnZXRzLApvcHRpbWlzZXIgc2V0dGluZ3MgYW5kIHRyYWluaW5nIGNvZGUgYXJlICppZGVudGljYWwqIGFjcm9zcyBtZXRob2RzOyBvbmx5IHRoZQphZGFwdGVyIHBhcmFtZXRlcmlzYXRpb24gYW5kIGl0cyByYW5rIGFsbG9jYXRpb24gLyBpbml0aWFsaXNhdGlvbiBkaWZmZXIuCgpNZXRob2RzCi0tLS0tLS0KZnVsbCAgICAgOiBmdWxsIGZpbmUtdHVuaW5nIChyZWZlcmVuY2UgdXBwZXIgYm91bmQgb24gdHJhaW5hYmxlIHBhcmFtZXRlcnMpCmxpbmVhciAgIDogbGluZWFyIHByb2JlIC0tIGNsYXNzaWZpY2F0aW9uIGhlYWQgb25seQpiaXRmaXQgICA6IGFsbCBiaWFzIHRlcm1zICsgaGVhZCAgICAgICAgICAgICAgICAgICAgICAoQmVuIFpha2VuIGV0IGFsLiwgMjAyMikKbG9yYSAgICAgOiB1bmlmb3JtIHJhbmsgYWNyb3NzIG1vZHVsZXMgICAgICAgICAgICAgICAgKEh1IGV0IGFsLiwgMjAyMikKZG9yYSAgICAgOiB3ZWlnaHQtZGVjb21wb3NlZCBMb1JBICAgICAgICAgICAgICAgICAgICAgKExpdSBldCBhbC4sIDIwMjQpCnBpc3NhICAgIDogTG9SQSBpbml0aWFsaXNlZCBmcm9tIHRoZSB0b3AtciBTVkQgb2YgVyAgIChNZW5nIGV0IGFsLiwgMjAyNCkKYWRhbG9yYSAgOiBTVkQgcGFyYW1ldGVyaXNhdGlvbiArIHRyYWluaW5nLXRpbWUgaW1wb3J0YW5jZSBwcnVuaW5nIChaaGFuZyBldCBhbC4sIDIwMjMpCmV2YSAgICAgIDogaW4tZG9tYWluIGFjdGl2YXRpb24gUENBIGluaXQgKyBleHBsYWluZWQtdmFyaWFuY2UgcmFuayByZWRpc3RyaWJ1dGlvbgogICAgICAgICAgIChQYWlzY2hlciBldCBhbC4sIDIwMjUpOyBleGFjdGx5IHRoZSB0YXUgPSAwIGNhc2Ugb2YgZHJpZnQKZHJpZnQgICAgOiBvdXJzIC0tIHJlZmVyZW5jZS1jb250cmFzdGl2ZSBkcmlmdCBzdWJzcGFjZSAoc2VlIGRyaWZ0LnB5KQoiIiIKaW1wb3J0IG1hdGgKaW1wb3J0IHJlCmltcG9ydCB0b3JjaAppbXBvcnQgdG9yY2gubm4gYXMgbm4KaW1wb3J0IHRvcmNoLm5uLmZ1bmN0aW9uYWwgYXMgRgoKCiMgPT09PT09PT09PT09PT09PT09PT09PT09PT09PT09PT09PT09PT09PT09PT09PT09PT09PT09PT09PT09PT09PT09PT09PT09PT0KIyBhZGFwdGVyIGxheWVycwojID09PT09PT09PT09PT09PT09PT09PT09PT09PT09PT09PT09PT09PT09PT09PT09PT09PT09PT09PT09PT09PT09PT09PT09PT09CmNsYXNzIExvUkFMaW5lYXIobm4uTW9kdWxlKToKICAgICIiIkZyb3plbiBiYXNlIGxpbmVhciArIHRyYWluYWJsZSBsb3ctcmFuayB1cGRhdGUgKG9wdGlvbmFsbHkgRG9SQS1zdHlsZSkuIiIiCgogICAgZGVmIF9faW5pdF9fKHNlbGYsIGJhc2U6IG5uLkxpbmVhciwgcjogaW50LCBhbHBoYTogZmxvYXQsIGRyb3BvdXQ6IGZsb2F0ID0gMC4wLAogICAgICAgICAgICAgICAgIHVzZV9kb3JhOiBib29sID0gRmFsc2UsIHNjYWxpbmdfbW9kZTogc3RyID0gImFscGhhX292ZXJfciIpOgogICAgICAgIHN1cGVyKCkuX19pbml0X18oKQogICAgICAgIHNlbGYuYmFzZSA9IGJhc2UKICAgICAgICBzZWxmLmJhc2Uud2VpZ2h0LnJlcXVpcmVzX2dyYWRfKEZhbHNlKQogICAgICAgIGlmIHNlbGYuYmFzZS5iaWFzIGlzIG5vdCBOb25lOgogICAgICAgICAgICBzZWxmLmJhc2UuYmlhcy5yZXF1aXJlc19ncmFkXyhGYWxzZSkKICAgICAgICBzZWxmLnIgPSBpbnQocikKICAgICAgICBzZWxmLnVzZV9kb3JhID0gdXNlX2RvcmEKICAgICAgICBzZWxmLmRyb3AgPSBubi5Ecm9wb3V0KGRyb3BvdXQpIGlmIGRyb3BvdXQgPiAwIGVsc2Ugbm4uSWRlbnRpdHkoKQogICAgICAgIGlmIHNlbGYuciA+IDA6CiAgICAgICAgICAgIHNlbGYubG9yYV9BID0gbm4uUGFyYW1ldGVyKHRvcmNoLnplcm9zKHNlbGYuciwgYmFzZS5pbl9mZWF0dXJlcykpCiAgICAgICAgICAgIHNlbGYubG9yYV9CID0gbm4uUGFyYW1ldGVyKHRvcmNoLnplcm9zKGJhc2Uub3V0X2ZlYXR1cmVzLCBzZWxmLnIpKQogICAgICAgICAgICBubi5pbml0LmthaW1pbmdfdW5pZm9ybV8oc2VsZi5sb3JhX0EsIGE9bWF0aC5zcXJ0KDUpKQogICAgICAgICAgICBpZiBzY2FsaW5nX21vZGUgPT0gIm9uZSI6CiAgICAgICAgICAgICAgICBzZWxmLnNjYWxpbmcgPSAxLjAKICAgICAgICAgICAgZWxpZiBzY2FsaW5nX21vZGUgPT0gInJzbG9yYSI6CiAgICAgICAgICAgICAgICBzZWxmLnNjYWxpbmcgPSBhbHBoYSAvIG1hdGguc3FydChzZWxmLnIpCiAgICAgICAgICAgIGVsc2U6CiAgICAgICAgICAgICAgICBzZWxmLnNjYWxpbmcgPSBhbHBoYSAvIHNlbGYucgogICAgICAgICAgICBpZiB1c2VfZG9yYToKICAgICAgICAgICAgICAgIHdpdGggdG9yY2gubm9fZ3JhZCgpOgogICAgICAgICAgICAgICAgICAgIG0gPSB0b3JjaC5saW5hbGcubm9ybShiYXNlLndlaWdodCwgZGltPTEpCiAgICAgICAgICAgICAgICBzZWxmLmRvcmFfbSA9IG5uLlBhcmFtZXRlcihtKQogICAgICAgIGVsc2U6CiAgICAgICAgICAgIHNlbGYuc2NhbGluZyA9IDAuMAoKICAgIGRlZiBkZWx0YV93KHNlbGYpOgogICAgICAgIHJldHVybiAoc2VsZi5sb3JhX0IgQCBzZWxmLmxvcmFfQSkgKiBzZWxmLnNjYWxpbmcKCiAgICBkZWYgZm9yd2FyZChzZWxmLCB4KToKICAgICAgICBpZiBzZWxmLnIgPT0gMDoKICAgICAgICAgICAgcmV0dXJuIHNlbGYuYmFzZSh4KQogICAgICAgIGlmIG5vdCBzZWxmLnVzZV9kb3JhOgogICAgICAgICAgICBvdXQgPSBzZWxmLmJhc2UoeCkKICAgICAgICAgICAgaCA9IHNlbGYuZHJvcCh4KSBAIHNlbGYubG9yYV9BLlQKICAgICAgICAgICAgcmV0dXJuIG91dCArIChoIEAgc2VsZi5sb3JhX0IuVCkgKiBzZWxmLnNjYWxpbmcKICAgICAgICB3ID0gc2VsZi5iYXNlLndlaWdodCArIHNlbGYuZGVsdGFfdygpCiAgICAgICAgbm9ybSA9IHRvcmNoLmxpbmFsZy5ub3JtKHcsIGRpbT0xKS5jbGFtcF9taW4oMWUtOCkuZGV0YWNoKCkKICAgICAgICB3ID0gdyAqIChzZWxmLmRvcmFfbSAvIG5vcm0pLnVuc3F1ZWV6ZSgxKQogICAgICAgIHJldHVybiBGLmxpbmVhcihzZWxmLmRyb3AoeCksIHcsIHNlbGYuYmFzZS5iaWFzKQoKCmNsYXNzIEFkYUxvUkFMaW5lYXIobm4uTW9kdWxlKToKICAgICIiIlNWRC1zdHlsZSBwYXJhbWV0ZXJpc2F0aW9uIGRXID0gUCBkaWFnKEUpIFEgdXNlZCBieSBBZGFMb1JBLgoKICAgIFRyaXBsZXRzIGFyZSBtYXNrZWQgKEVfaSA8LSAwKSBieSB0aGUgZ2xvYmFsIGJ1ZGdldCBjb250cm9sbGVyOyBtYXNrZWQgdHJpcGxldHMKICAgIHN0b3AgY29udHJpYnV0aW5nIGJ1dCBzdGF5IGFsbG9jYXRlZCB1bnRpbCB0aGUgc2NoZWR1bGUgZW5kcywgZXhhY3RseSBhcyBpbiB0aGUKICAgIG9yaWdpbmFsIGZvcm11bGF0aW9uLgogICAgIiIiCgogICAgZGVmIF9faW5pdF9fKHNlbGYsIGJhc2U6IG5uLkxpbmVhciwgcjogaW50LCBhbHBoYTogZmxvYXQsIGRyb3BvdXQ6IGZsb2F0ID0gMC4wKToKICAgICAgICBzdXBlcigpLl9faW5pdF9fKCkKICAgICAgICBzZWxmLmJhc2UgPSBiYXNlCiAgICAgICAgc2VsZi5iYXNlLndlaWdodC5yZXF1aXJlc19ncmFkXyhGYWxzZSkKICAgICAgICBpZiBzZWxmLmJhc2UuYmlhcyBpcyBub3QgTm9uZToKICAgICAgICAgICAgc2VsZi5iYXNlLmJpYXMucmVxdWlyZXNfZ3JhZF8oRmFsc2UpCiAgICAgICAgc2VsZi5yID0gaW50KHIpCiAgICAgICAgc2VsZi5kcm9wID0gbm4uRHJvcG91dChkcm9wb3V0KSBpZiBkcm9wb3V0ID4gMCBlbHNlIG5uLklkZW50aXR5KCkKICAgICAgICBzZWxmLmxvcmFfUCA9IG5uLlBhcmFtZXRlcih0b3JjaC56ZXJvcyhiYXNlLm91dF9mZWF0dXJlcywgc2VsZi5yKSkKICAgICAgICBzZWxmLmxvcmFfRSA9IG5uLlBhcmFtZXRlcih0b3JjaC56ZXJvcyhzZWxmLnIpKQogICAgICAgIHNlbGYubG9yYV9RID0gbm4uUGFyYW1ldGVyKHRvcmNoLnplcm9zKHNlbGYuciwgYmFzZS5pbl9mZWF0dXJlcykpCiAgICAgICAgbm4uaW5pdC5ub3JtYWxfKHNlbGYubG9yYV9QLCBzdGQ9MC4wMikKICAgICAgICBubi5pbml0Lm5vcm1hbF8oc2VsZi5sb3JhX1EsIHN0ZD0wLjAyKQogICAgICAgIG5uLmluaXQuemVyb3NfKHNlbGYubG9yYV9FKQogICAgICAgIHNlbGYuc2NhbGluZyA9IGFscGhhIC8gc2VsZi5yCiAgICAgICAgc2VsZi5yZWdpc3Rlcl9idWZmZXIoIm1hc2siLCB0b3JjaC5vbmVzKHNlbGYucikpCgogICAgZGVmIGZvcndhcmQoc2VsZiwgeCk6CiAgICAgICAgb3V0ID0gc2VsZi5iYXNlKHgpCiAgICAgICAgZSA9IHNlbGYubG9yYV9FICogc2VsZi5tYXNrCiAgICAgICAgaCA9IHNlbGYuZHJvcCh4KSBAIHNlbGYubG9yYV9RLlQKICAgICAgICBoID0gaCAqIGUKICAgICAgICByZXR1cm4gb3V0ICsgKGggQCBzZWxmLmxvcmFfUC5UKSAqIHNlbGYuc2NhbGluZwoKICAgIGRlZiBvcnRob19wZW5hbHR5KHNlbGYpOgogICAgICAgIHAsIHEgPSBzZWxmLmxvcmFfUCwgc2VsZi5sb3JhX1EKICAgICAgICBpcCA9IHAuVCBAIHAKICAgICAgICBpcSA9IHEgQCBxLlQKICAgICAgICBleWUgPSB0b3JjaC5leWUoc2VsZi5yLCBkZXZpY2U9cC5kZXZpY2UsIGR0eXBlPXAuZHR5cGUpCiAgICAgICAgcmV0dXJuICgoaXAgLSBleWUpICoqIDIpLnN1bSgpICsgKChpcSAtIGV5ZSkgKiogMikuc3VtKCkKCgojID09PT09PT09PT09PT09PT09PT09PT09PT09PT09PT09PT09PT09PT09PT09PT09PT09PT09PT09PT09PT09PT09PT09PT09PT09CiMgaW5qZWN0aW9uIGhlbHBlcnMKIyA9PT09PT09PT09PT09PT09PT09PT09PT09PT09PT09PT09PT09PT09PT09PT09PT09PT09PT09PT09PT09PT09PT09PT09PT09PQpkZWYgX2dldF9wYXJlbnQobW9kZWwsIG5hbWUpOgogICAgcGFydHMgPSBuYW1lLnNwbGl0KCIuIikKICAgIHBhcmVudCA9IG1vZGVsCiAgICBmb3IgcCBpbiBwYXJ0c1s6LTFdOgogICAgICAgIHBhcmVudCA9IGdldGF0dHIocGFyZW50LCBwKQogICAgcmV0dXJuIHBhcmVudCwgcGFydHNbLTFdCgoKZGVmIGluamVjdF9hZGFwdGVycyhtb2RlbCwgcmFua3MsIGFscGhhPTE2LjAsIGRyb3BvdXQ9MC4wLCBraW5kPSJsb3JhIiwKICAgICAgICAgICAgICAgICAgICBzY2FsaW5nX21vZGU9ImFscGhhX292ZXJfciIpOgogICAgIiIiUmVwbGFjZSB0aGUgbmFtZWQgbm4uTGluZWFyIG1vZHVsZXMgd2l0aCBhZGFwdGVyLXdyYXBwZWQgdmVyc2lvbnMuIiIiCiAgICBpbmplY3RlZCA9IHt9CiAgICBmb3IgbmFtZSwgciBpbiByYW5rcy5pdGVtcygpOgogICAgICAgIGlmIHIgPD0gMDoKICAgICAgICAgICAgY29udGludWUKICAgICAgICBwYXJlbnQsIGF0dHIgPSBfZ2V0X3BhcmVudChtb2RlbCwgbmFtZSkKICAgICAgICBiYXNlID0gZ2V0YXR0cihwYXJlbnQsIGF0dHIpCiAgICAgICAgaWYga2luZCA9PSAiYWRhbG9yYSI6CiAgICAgICAgICAgIG5ldyA9IEFkYUxvUkFMaW5lYXIoYmFzZSwgciwgYWxwaGEsIGRyb3BvdXQpCiAgICAgICAgZWxzZToKICAgICAgICAgICAgbmV3ID0gTG9SQUxpbmVhcihiYXNlLCByLCBhbHBoYSwgZHJvcG91dCwKICAgICAgICAgICAgICAgICAgICAgICAgICAgICB1c2VfZG9yYT0oa2luZCA9PSAiZG9yYSIpLCBzY2FsaW5nX21vZGU9c2NhbGluZ19tb2RlKQogICAgICAgIHNldGF0dHIocGFyZW50LCBhdHRyLCBuZXcpCiAgICAgICAgaW5qZWN0ZWRbbmFtZV0gPSBuZXcKICAgIHJldHVybiBpbmplY3RlZAoKCmRlZiBmcmVlemVfYmFja2JvbmUobW9kZWwsIGhlYWRfcHJlZml4ZXM9KCJjbGFzc2lmaWVyIiwgInNjb3JlIiwgImhlYWQiKSk6CiAgICBmb3IgbmFtZSwgcCBpbiBtb2RlbC5uYW1lZF9wYXJhbWV0ZXJzKCk6CiAgICAgICAgcC5yZXF1aXJlc19ncmFkXyhhbnkobmFtZS5zdGFydHN3aXRoKGgpIG9yICgiLiIgKyBoICsgIi4iKSBpbiBuYW1lCiAgICAgICAgICAgICAgICAgICAgICAgICAgICAgZm9yIGggaW4gaGVhZF9wcmVmaXhlcykpCgoKZGVmIHVuZnJlZXplX2hlYWQobW9kZWwsIGhlYWRfcHJlZml4ZXM9KCJjbGFzc2lmaWVyIiwgInNjb3JlIiwgImhlYWQiKSk6CiAgICBmb3IgbmFtZSwgcCBpbiBtb2RlbC5uYW1lZF9wYXJhbWV0ZXJzKCk6CiAgICAgICAgaWYgYW55KGggaW4gbmFtZS5zcGxpdCgiLiIpIGZvciBoIGluIGhlYWRfcHJlZml4ZXMpOgogICAgICAgICAgICBwLnJlcXVpcmVzX2dyYWRfKFRydWUpCgoKZGVmIHNldF90cmFpbmFibGVfYWRhcHRlcnMobW9kZWwpOgogICAgZm9yIG5hbWUsIHAgaW4gbW9kZWwubmFtZWRfcGFyYW1ldGVycygpOgogICAgICAgIGlmIGFueShrIGluIG5hbWUgZm9yIGsgaW4gKCJsb3JhX0EiLCAibG9yYV9CIiwgImxvcmFfUCIsICJsb3JhX0UiLAogICAgICAgICAgICAgICAgICAgICAgICAgICAgICAgICAgICJsb3JhX1EiLCAiZG9yYV9tIikpOgogICAgICAgICAgICBwLnJlcXVpcmVzX2dyYWRfKFRydWUpCgoKIyA9PT09PT09PT09PT09PT09PT09PT09PT09PT09PT09PT09PT09PT09PT09PT09PT09PT09PT09PT09PT09PT09PT09PT09PT09PQojIGluaXRpYWxpc2F0aW9uIHNjaGVtZXMKIyA9PT09PT09PT09PT09PT09PT09PT09PT09PT09PT09PT09PT09PT09PT09PT09PT09PT09PT09PT09PT09PT09PT09PT09PT09PQpAdG9yY2gubm9fZ3JhZCgpCmRlZiBpbml0X3Bpc3NhKGxheWVyOiBMb1JBTGluZWFyKToKICAgICIiIkEgPSBzcXJ0KFNfcikgVl9yXlQsIEIgPSBVX3Igc3FydChTX3IpOyByZXNpZHVhbCB3ZWlnaHQgVyAtIEJBIHN0YXlzIGZyb3plbi4iIiIKICAgIHcgPSBsYXllci5iYXNlLndlaWdodC5kYXRhLmRvdWJsZSgpCiAgICB1LCBzLCB2aCA9IHRvcmNoLmxpbmFsZy5zdmQodywgZnVsbF9tYXRyaWNlcz1GYWxzZSkKICAgIHIgPSBsYXllci5yCiAgICB1ciwgc3IsIHZyID0gdVs6LCA6cl0sIHNbOnJdLCB2aFs6ciwgOl0KICAgIHNxID0gdG9yY2guc3FydChzcikKICAgIGEgPSAodG9yY2guZGlhZyhzcSkgQCB2cikKICAgIGIgPSAodXIgQCB0b3JjaC5kaWFnKHNxKSkKICAgIGxheWVyLmxvcmFfQS5kYXRhLmNvcHlfKGEudG8obGF5ZXIubG9yYV9BLmR0eXBlKSkKICAgIGxheWVyLmxvcmFfQi5kYXRhLmNvcHlfKGIudG8obGF5ZXIubG9yYV9CLmR0eXBlKSkKICAgIGxheWVyLnNjYWxpbmcgPSAxLjAKICAgIGxheWVyLmJhc2Uud2VpZ2h0LmRhdGEuY29weV8oKHcgLSBiIEAgYSkudG8obGF5ZXIuYmFzZS53ZWlnaHQuZHR5cGUpKQoKCkB0b3JjaC5ub19ncmFkKCkKZGVmIGluaXRfc3Vic3BhY2UobGF5ZXI6IExvUkFMaW5lYXIsIGJhc2lzOiB0b3JjaC5UZW5zb3IpOgogICAgIiIiQSA8LSBsZWFkaW5nIHN1YnNwYWNlIGRpcmVjdGlvbnMgKHJvd3MpLCBCIDwtIDAgc28gZFcgPSAwIGF0IGluaXRpYWxpc2F0aW9uLgoKICAgIGJhc2lzOiAoZF9pbiwgaykgY29sdW1uLW9ydGhvbm9ybWFsLCBrID49IGxheWVyLnIgaW4gbm9ybWFsIG9wZXJhdGlvbi4KCiAgICBJZiB0aGUgY2FjaGVkIGJhc2lzIGhhcyBmZXdlciBjb2x1bW5zIHRoYW4gdGhlIGFsbG9jYXRlZCByYW5rLCB0aGUgc3VycGx1cwogICAgcm93cyBhcmUgZmlsbGVkIHdpdGggcmFuZG9tIGRpcmVjdGlvbnMgb3J0aG9nb25hbGlzZWQgYWdhaW5zdCB0aGUgYmFzaXMgLS0KICAgIG5ldmVyIGxlZnQgYXMgemVyb3MsIHdoaWNoIHdvdWxkIG1ha2UgdGhvc2UgcmFua3MgcGVybWFuZW50bHkgZGVhZCAoYSB6ZXJvCiAgICByb3cgb2YgQSBnaXZlcyBhIHplcm8gZ3JhZGllbnQgdG8gdGhlIGNvcnJlc3BvbmRpbmcgY29sdW1uIG9mIEIpLgogICAgIiIiCiAgICByID0gbWluKGxheWVyLnIsIGJhc2lzLnNoYXBlWzFdKQogICAgbGF5ZXIubG9yYV9BLmRhdGEuemVyb18oKQogICAgbGF5ZXIubG9yYV9BLmRhdGFbOnJdLmNvcHlfKGJhc2lzWzosIDpyXS5ULnRvKGxheWVyLmxvcmFfQS5kdHlwZSkpCiAgICBpZiBsYXllci5yID4gcjoKICAgICAgICBleHRyYSA9IHRvcmNoLnJhbmRuKGxheWVyLmJhc2UuaW5fZmVhdHVyZXMsIGxheWVyLnIgLSByLAogICAgICAgICAgICAgICAgICAgICAgICAgICAgZHR5cGU9YmFzaXMuZHR5cGUpCiAgICAgICAgZXh0cmEgLT0gYmFzaXNbOiwgOnJdIEAgKGJhc2lzWzosIDpyXS5UIEAgZXh0cmEpCiAgICAgICAgcSwgXyA9IHRvcmNoLmxpbmFsZy5xcihleHRyYSkKICAgICAgICBsYXllci5sb3JhX0EuZGF0YVtyOl0uY29weV8ocS5ULnRvKGxheWVyLmxvcmFfQS5kdHlwZSkpCiAgICBsYXllci5sb3JhX0IuZGF0YS56ZXJvXygpCgoKIyA9PT09PT09PT09PT09PT09PT09PT09PT09PT09PT09PT09PT09PT09PT09PT09PT09PT09PT09PT09PT09PT09PT09PT09PT09PQojIEFkYUxvUkEgYnVkZ2V0IGNvbnRyb2xsZXIKIyA9PT09PT09PT09PT09PT09PT09PT09PT09PT09PT09PT09PT09PT09PT09PT09PT09PT09PT09PT09PT09PT09PT09PT09PT09PQpjbGFzcyBBZGFMb1JBQ29udHJvbGxlcjoKICAgICIiIkdsb2JhbCBpbXBvcnRhbmNlLWJhc2VkIGJ1ZGdldCBzY2hlZHVsZXIgKFpoYW5nIGV0IGFsLiwgSUNMUiAyMDIzKS4KCiAgICBJbXBvcnRhbmNlIG9mIHRyaXBsZXQgaSBjb21iaW5lcyB0aGUgc2Vuc2l0aXZpdHkgb2YgRV9pIGFuZCBvZiB0aGUgY29ycmVzcG9uZGluZwogICAgcm93L2NvbHVtbiBvZiBQIGFuZCBRLCBlYWNoIHNtb290aGVkIGJ5IGFuIGV4cG9uZW50aWFsIG1vdmluZyBhdmVyYWdlLCBwbHVzIGFuCiAgICB1bmNlcnRhaW50eSB0ZXJtLiAgVGhlIHRvdGFsIGJ1ZGdldCBmb2xsb3dzIGEgY3ViaWMgc2NoZWR1bGUgZnJvbSBiX2luaXQgdG8KICAgIGJfdGFyZ2V0OyB0aGUgbG93ZXN0LWltcG9ydGFuY2UgdHJpcGxldHMgYXJlIG1hc2tlZC4KICAgICIiIgoKICAgIGRlZiBfX2luaXRfXyhzZWxmLCBsYXllcnMsIHRhcmdldF9yYW5rX3RvdGFsLCBpbml0X3JhbmtfdG90YWwsCiAgICAgICAgICAgICAgICAgdG90YWxfc3RlcHMsIHdhcm11cF9mcmFjPTAuMSwgZmluYWxfZnJhYz0wLjc1LAogICAgICAgICAgICAgICAgIGJldGExPTAuODUsIGJldGEyPTAuODUpOgogICAgICAgIHNlbGYubGF5ZXJzID0gbGF5ZXJzCiAgICAgICAgc2VsZi5iX3RhcmdldCA9IHRhcmdldF9yYW5rX3RvdGFsCiAgICAgICAgc2VsZi5iX2luaXQgPSBpbml0X3JhbmtfdG90YWwKICAgICAgICBzZWxmLnRpID0gaW50KHdhcm11cF9mcmFjICogdG90YWxfc3RlcHMpCiAgICAgICAgc2VsZi50ZiA9IGludChmaW5hbF9mcmFjICogdG90YWxfc3RlcHMpCiAgICAgICAgc2VsZi50b3RhbF9zdGVwcyA9IHRvdGFsX3N0ZXBzCiAgICAgICAgc2VsZi5iZXRhMSwgc2VsZi5iZXRhMiA9IGJldGExLCBiZXRhMgogICAgICAgIHNlbGYuaXB0LCBzZWxmLmV4cF9pcHQsIHNlbGYuZXhwX3VuYyA9IHt9LCB7fSwge30KCiAgICBkZWYgX3Njb3JlKHNlbGYsIGxheWVyLCBuYW1lKToKICAgICAgICB3aXRoIHRvcmNoLm5vX2dyYWQoKToKICAgICAgICAgICAgcGFydHMgPSBbXQogICAgICAgICAgICBmb3IgcCBpbiAobGF5ZXIubG9yYV9FLCBsYXllci5sb3JhX1AsIGxheWVyLmxvcmFfUSk6CiAgICAgICAgICAgICAgICBpZiBwLmdyYWQgaXMgTm9uZToKICAgICAgICAgICAgICAgICAgICByZXR1cm4gTm9uZQogICAgICAgICAgICAgICAgcyA9IChwICogcC5ncmFkKS5hYnMoKS5kZXRhY2goKQogICAgICAgICAgICAgICAgcGFydHMuYXBwZW5kKHMpCiAgICAgICAgICAgIGVfcyA9IHBhcnRzWzBdICAgICAgICAgICAgICAgICAgICAgICAgICAgICAgICAgICAjIChyLCkKICAgICAgICAgICAgcF9zID0gcGFydHNbMV0ubWVhbihkaW09MCkgICAgICAgICAgICAgICAgICAgICAgICMgKHIsKQogICAgICAgICAgICBxX3MgPSBwYXJ0c1syXS5tZWFuKGRpbT0xKSAgICAgICAgICAgICAgICAgICAgICAgIyAociwpCiAgICAgICAgICAgIHJhdyA9IGVfcyArIHBfcyArIHFfcwogICAgICAgICAgICBpZiBuYW1lIG5vdCBpbiBzZWxmLmV4cF9pcHQ6CiAgICAgICAgICAgICAgICBzZWxmLmV4cF9pcHRbbmFtZV0gPSB0b3JjaC56ZXJvc19saWtlKHJhdykKICAgICAgICAgICAgICAgIHNlbGYuZXhwX3VuY1tuYW1lXSA9IHRvcmNoLnplcm9zX2xpa2UocmF3KQogICAgICAgICAgICBzZWxmLmV4cF9pcHRbbmFtZV0gPSBzZWxmLmJldGExICogc2VsZi5leHBfaXB0W25hbWVdICsgKDEgLSBzZWxmLmJldGExKSAqIHJhdwogICAgICAgICAgICBzZWxmLmV4cF91bmNbbmFtZV0gPSAoc2VsZi5iZXRhMiAqIHNlbGYuZXhwX3VuY1tuYW1lXQogICAgICAgICAgICAgICAgICAgICAgICAgICAgICAgICAgKyAoMSAtIHNlbGYuYmV0YTIpICogKHJhdyAtIHNlbGYuZXhwX2lwdFtuYW1lXSkuYWJzKCkpCiAgICAgICAgICAgIHJldHVybiBzZWxmLmV4cF9pcHRbbmFtZV0gKiBzZWxmLmV4cF91bmNbbmFtZV0KCiAgICBkZWYgYnVkZ2V0KHNlbGYsIHN0ZXApOgogICAgICAgIGlmIHN0ZXAgPD0gc2VsZi50aToKICAgICAgICAgICAgcmV0dXJuIHNlbGYuYl9pbml0CiAgICAgICAgaWYgc3RlcCA+PSBzZWxmLnRmOgogICAgICAgICAgICByZXR1cm4gc2VsZi5iX3RhcmdldAogICAgICAgIGZyYWMgPSAxLjAgLSAoc3RlcCAtIHNlbGYudGkpIC8gbWF4KHNlbGYudGYgLSBzZWxmLnRpLCAxKQogICAgICAgIHJldHVybiBpbnQoc2VsZi5iX3RhcmdldCArIChzZWxmLmJfaW5pdCAtIHNlbGYuYl90YXJnZXQpICogKGZyYWMgKiogMykpCgogICAgZGVmIHN0ZXAoc2VsZiwgZ2xvYmFsX3N0ZXApOgogICAgICAgIHNjb3JlcyA9IHt9CiAgICAgICAgZm9yIG5hbWUsIGxheWVyIGluIHNlbGYubGF5ZXJzLml0ZW1zKCk6CiAgICAgICAgICAgIHMgPSBzZWxmLl9zY29yZShsYXllciwgbmFtZSkKICAgICAgICAgICAgaWYgcyBpcyBOb25lOgogICAgICAgICAgICAgICAgcmV0dXJuCiAgICAgICAgICAgIHNjb3Jlc1tuYW1lXSA9IHMKICAgICAgICBiID0gc2VsZi5idWRnZXQoZ2xvYmFsX3N0ZXApCiAgICAgICAgYWxsdiA9IHRvcmNoLmNhdChbdiBmb3IgdiBpbiBzY29yZXMudmFsdWVzKCldKQogICAgICAgIGsgPSBtYXgoaW50KGIpLCAxKQogICAgICAgIGlmIGsgPj0gYWxsdi5udW1lbCgpOgogICAgICAgICAgICBmb3IgbGF5ZXIgaW4gc2VsZi5sYXllcnMudmFsdWVzKCk6CiAgICAgICAgICAgICAgICBsYXllci5tYXNrLmZpbGxfKDEuMCkKICAgICAgICAgICAgcmV0dXJuCiAgICAgICAgdGhyZXNoID0gdG9yY2gudG9wayhhbGx2LCBrLCBsYXJnZXN0PVRydWUpLnZhbHVlcy5taW4oKQogICAgICAgIGZvciBuYW1lLCBsYXllciBpbiBzZWxmLmxheWVycy5pdGVtcygpOgogICAgICAgICAgICBsYXllci5tYXNrLmNvcHlfKChzY29yZXNbbmFtZV0gPj0gdGhyZXNoKS50byhsYXllci5tYXNrLmR0eXBlKSkKCiAgICBkZWYgYWN0aXZlX3JhbmtfdG90YWwoc2VsZik6CiAgICAgICAgcmV0dXJuIGludChzdW0obC5tYXNrLnN1bSgpLml0ZW0oKSBmb3IgbCBpbiBzZWxmLmxheWVycy52YWx1ZXMoKSkpCg==", "engine.py": "...", "profile_drift.py": "IiIiQ29tcHV0ZSBhbmQgY2FjaGUgRFJJRlQgcHJvZmlsZXMgZm9yIGEgKG1vZGVsLCB0YXNrKSBwYWlyLgoKT25lIGZvcndhcmQtb25seSBwYXNzIG92ZXIgYSBnZW5lcmFsLWRvbWFpbiByZWZlcmVuY2UgY29ycHVzIGFuZCBvbmUgb3ZlciB0aGUKdW5sYWJlbGxlZCB0YXNrIHRleHQgeWllbGRzLCBwZXIgYWRhcHRhYmxlIGxpbmVhciBtb2R1bGUsIHRoZSBzZWNvbmQtbW9tZW50Cm1hdHJpY2VzIFNpZ21hX0cgYW5kIFNpZ21hX0QuICBGb3IgZWFjaCByZXF1ZXN0ZWQgdGF1IHdlIHRoZW4gc3RvcmUKCiAgICAtIHRoZSBmdWxsIGRyaWZ0IHNwZWN0cnVtIChlaWdlbnZhbHVlcyBvZiBTaWdtYX4gaW4gZGVzY2VuZGluZyBvcmRlciksCiAgICAtIHRoZSBsZWFkaW5nIHJfbWF4IGRyaWZ0IGVpZ2VudmVjdG9ycyAodGhlIGFkYXB0ZXIgaW5pdGlhbGlzYXRpb24gYmFzaXMpLAogICAgLSB0cmFjZShTaWdtYV9EKSBhbmQgdGhlIHJlZmVyZW5jZSBzdWJzcGFjZSBkaW1lbnNpb24gay4KCnRhdSA9IDAgcmVkdWNlcyB0byBwbGFpbiBpbi1kb21haW4gYWN0aXZhdGlvbiBQQ0EsIGkuZS4gdGhlIEVWQSBiYXNlbGluZS4KClVzYWdlOgogICAgcHl0aG9uIHNyYy9wcm9maWxlX2RyaWZ0LnB5IC0tbW9kZWwgcm9iZXJ0YS1iYXNlIC0tdGFzayBjaGVtcHJvdAoiIiIKaW1wb3J0IGFyZ3BhcnNlCmltcG9ydCBqc29uCmltcG9ydCBvcwppbXBvcnQgc3lzCmltcG9ydCB0aW1lCgppbXBvcnQgdG9yY2gKCnN5cy5wYXRoLmluc2VydCgwLCBvcy5wYXRoLmRpcm5hbWUob3MucGF0aC5hYnNwYXRoKF9fZmlsZV9fKSkpCmltcG9ydCBkYXRhIGFzIGRhdGFfbW9kICAgICAgICAgICMgbm9xYTogRTQwMgppbXBvcnQgZHJpZnQgYXMgZHJpZnRfbW9kICAgICAgICAjIG5vcWE6IEU0MDIKZnJvbSBydW5zcGVjIGltcG9ydCBwcm9maWxlX2tleSAgIyBub3FhOiBFNDAyLEY0MDEgICh0b3JjaC1mcmVlLCBzaGFyZWQgd2l0aCBncmlkLnB5KQoKUk9PVCA9IG9zLnBhdGguZGlybmFtZShvcy5wYXRoLmRpcm5hbWUob3MucGF0aC5hYnNwYXRoKF9fZmlsZV9fKSkpClBST0ZJTEVfRElSID0gb3MucGF0aC5qb2luKFJPT1QsICJydW5zIiwgInByb2ZpbGVzIikKCgpkZWYgbGVhZF9zdW1tYXJ5KGJhc2lzLCBldmFscywgdHJhY2VfZCwgZF9pbik6CiAgICAiIiJMZWFkaW5nIGRpcmVjdGlvbiBvZiBvbmUgbW9kdWxlOiBpdHMgbGFyZ2VzdCBjb29yZGluYXRlLCB0aGUgd2VpZ2h0IGl0CiAgICBwdXRzIHRoZXJlLCBhbmQgaXRzIGVuZXJneSByZWxhdGl2ZSB0byBhbiBhdmVyYWdlIGRpcmVjdGlvbi4iIiIKICAgIHUgPSB0b3JjaC5hc190ZW5zb3IoYmFzaXMpWzosIDBdLmFicygpCiAgICBsYW1fYmFyID0gdHJhY2VfZCAvIGRfaW4KICAgIHJldHVybiB7ImFyZ21heCI6IGludCh1LmFyZ21heCgpKSwgInBlYWsiOiBmbG9hdCh1Lm1heCgpKSwKICAgICAgICAgICAgImVuZXJneV9yZWwiOiBmbG9hdChldmFsc1swXSkgLyBtYXgobGFtX2JhciwgMWUtMzApfQoKCmRlZiBtYWluKCk6CiAgICBhcCA9IGFyZ3BhcnNlLkFyZ3VtZW50UGFyc2VyKCkKICAgIGFwLmFkZF9hcmd1bWVudCgiLS1tb2RlbCIsIGRlZmF1bHQ9InJvYmVydGEtYmFzZSIpCiAgICBhcC5hZGRfYXJndW1lbnQoIi0tdGFzayIsIGRlZmF1bHQ9ImNoZW1wcm90IikKICAgIGFwLmFkZF9hcmd1bWVudCgiLS1uX3JlZiIsIHR5cGU9aW50LCBkZWZhdWx0PTEwMjQpCiAgICBhcC5hZGRfYXJndW1lbnQoIi0tbl9kb20iLCB0eXBlPWludCwgZGVmYXVsdD0xMDI0KQogICAgYXAuYWRkX2FyZ3VtZW50KCItLXRhdXMiLCBkZWZhdWx0PSIwLjAsMC41LDAuOSwwLjk1LDAuOTkiKQogICAgYXAuYWRkX2FyZ3VtZW50KCItLXJfbWF4IiwgdHlwZT1pbnQsIGRlZmF1bHQ9NjQpCiAgICBhcC5hZGRfYXJndW1lbnQoIi0tYmF0Y2hfc2l6ZSIsIHR5cGU9aW50LCBkZWZhdWx0PTE2KQogICAgYXAuYWRkX2FyZ3VtZW50KCItLW1heF9sZW4iLCB0eXBlPWludCwgZGVmYXVsdD1Ob25lKQogICAgYXAuYWRkX2FyZ3VtZW50KCItLWZvcmNlIiwgYWN0aW9uPSJzdG9yZV90cnVlIikKICAgIGFwLmFkZF9hcmd1bWVudCgiLS1vdXRfc3VmZml4IiwgZGVmYXVsdD0iIiwKICAgICAgICAgICAgICAgICAgICBoZWxwPSJ3cml0ZSB0byBhIHNlcGFyYXRlbHkgbmFtZWQgcHJvZmlsZSwgZS5nLiB0byB0aW1lIGEgIgogICAgICAgICAgICAgICAgICAgICAgICAgInNpbmdsZS10YXUgcnVuIHdpdGhvdXQgdG91Y2hpbmcgdGhlIGNhY2hlZCBzd2VlcCIpCiAgICBhcC5hZGRfYXJndW1lbnQoIi0tcmF3IiwgYWN0aW9uPSJzdG9yZV90cnVlIiwKICAgICAgICAgICAgICAgICAgICBoZWxwPSJyYXcgc2Vjb25kIG1vbWVudHMgaW5zdGVhZCBvZiBjb3ZhcmlhbmNlcyIpCiAgICBhcC5hZGRfYXJndW1lbnQoIi0tcmVmIiwgZGVmYXVsdD0id2lraXRleHQiLCBjaG9pY2VzPWxpc3QoZGF0YV9tb2QuUkVGRVJFTkNFX0tJTkRTKSwKICAgICAgICAgICAgICAgICAgICBoZWxwPSJyZWZlcmVuY2UgY29ycHVzOiBXaWtpVGV4dC0xMDMgKGRlZmF1bHQpLCBDTk4vRGFpbHlNYWlsICIKICAgICAgICAgICAgICAgICAgICAgICAgICJuZXdzLCB3b3JkLXNodWZmbGVkIFdpa2lUZXh0LCBvciB1bmlmb3JtbHkgcmFuZG9tIHRva2VucyIpCiAgICBhcC5hZGRfYXJndW1lbnQoIi0tZ2V2IiwgYWN0aW9uPSJzdG9yZV90cnVlIiwKICAgICAgICAgICAgICAgICAgICBoZWxwPSJhbHNvIHN0b3JlIHRoZSBnZW5lcmFsaXNlZCBlaWdlbnZlY3RvcnMgb2YgIgogICAgICAgICAgICAgICAgICAgICAgICAgIihTaWdtYV9ELCBTaWdtYV9HKSBmb3IgdGhlIEdFViBpbml0aWFsaXNhdGlvbiIpCiAgICBhcC5hZGRfYXJndW1lbnQoIi0tZ2V2X3NocmluayIsIHR5cGU9ZmxvYXQsIGRlZmF1bHQ9MC4xKQogICAgYXJncyA9IGFwLnBhcnNlX2FyZ3MoKQoKICAgIGZyb20gdHJhbnNmb3JtZXJzIGltcG9ydCBBdXRvVG9rZW5pemVyLCBBdXRvTW9kZWxGb3JTZXF1ZW5jZUNsYXNzaWZpY2F0aW9uCgogICAgb3MubWFrZWRpcnMoUFJPRklMRV9ESVIsIGV4aXN0X29rPVRydWUpCiAgICBjZW50ZXIgPSBub3QgYXJncy5yYXcKICAgIGtleSA9IHByb2ZpbGVfa2V5KGFyZ3MubW9kZWwsIGFyZ3MudGFzaywgYXJncy5uX3JlZiwgYXJncy5uX2RvbSwKICAgICAgICAgICAgICAgICAgICAgIGNlbnRlcj1jZW50ZXIsIHJlZj1hcmdzLnJlZiwgZ2V2PWFyZ3MuZ2V2KSArIGFyZ3Mub3V0X3N1ZmZpeAogICAgb3V0X3BhdGggPSBvcy5wYXRoLmpvaW4oUFJPRklMRV9ESVIsIGtleSArICIucHQiKQogICAgaWYgb3MucGF0aC5leGlzdHMob3V0X3BhdGgpIGFuZCBub3QgYXJncy5mb3JjZToKICAgICAgICBwcmludCgicHJvZmlsZSBleGlzdHM6Iiwgb3V0X3BhdGgpCiAgICAgICAgcmV0dXJuCgogICAgZGV2aWNlID0gdG9yY2guZGV2aWNlKCJjdWRhIiBpZiB0b3JjaC5jdWRhLmlzX2F2YWlsYWJsZSgpIGVsc2UgImNwdSIpCiAgICB0YXNrID0gZGF0YV9tb2QubG9hZF90YXNrKGFyZ3MudGFzaykKICAgIG1heF9sZW4gPSBhcmdzLm1heF9sZW4gb3IgZGF0YV9tb2QuVEFTS19NQVhMRU4uZ2V0KGFyZ3MudGFzaywgMTI4KQoKICAgIHRvayA9IEF1dG9Ub2tlbml6ZXIuZnJvbV9wcmV0cmFpbmVkKGFyZ3MubW9kZWwpCiAgICAjIHByb2ZpbGUgaW4gZnAzMiB3aGF0ZXZlciB0aGUgY2hlY2twb2ludCdzIHN0b3JlZCBkdHlwZSAoYmYxNiBmb3IgU21vbExNMikKICAgIG1vZGVsID0gQXV0b01vZGVsRm9yU2VxdWVuY2VDbGFzc2lmaWNhdGlvbi5mcm9tX3ByZXRyYWluZWQoCiAgICAgICAgYXJncy5tb2RlbCwgbnVtX2xhYmVscz10YXNrWyJudW1fbGFiZWxzIl0pLmZsb2F0KCkudG8oZGV2aWNlKQogICAgZHJpZnRfbW9kLmVuc3VyZV9wYWRkaW5nKHRvaywgbW9kZWwpCiAgICBtb2RlbC5ldmFsKCkKCiAgICBtb2R1bGVzID0gZHJpZnRfbW9kLmZpbmRfdGFyZ2V0X21vZHVsZXMobW9kZWwpCiAgICBwcmludChmIntsZW4obW9kdWxlcyl9IGFkYXB0YWJsZSBtb2R1bGVzIikKCiAgICBkb21fdGV4dHMgPSB0YXNrWyJzcGxpdHMiXVsidHJhaW4iXVswXVs6YXJncy5uX2RvbV0KICAgIHJlZl90ZXh0cyA9IGRhdGFfbW9kLmxvYWRfcmVmZXJlbmNlX2NvcnB1cyhuX2RvY3M9YXJncy5uX3JlZiwga2luZD1hcmdzLnJlZiwKICAgICAgICAgICAgICAgICAgICAgICAgICAgICAgICAgICAgICAgICAgICAgICB0b2tlbml6ZXI9dG9rLCBuX3Rva2Vucz1tYXhfbGVuIC0gMikKICAgIHByaW50KGYicmVmZXJlbmNlIHtsZW4ocmVmX3RleHRzKX0gZG9jcyAoe2FyZ3MucmVmfSkgfCBkb21haW4ge2xlbihkb21fdGV4dHMpfSBkb2NzICIKICAgICAgICAgIGYifCBtYXhfbGVuIHttYXhfbGVufSIpCgogICAgdDAgPSB0aW1lLnBlcmZfY291bnRlcigpCiAgICBjb3ZfZyA9IGRyaWZ0X21vZC5jb2xsZWN0X2NvdmFyaWFuY2VzKG1vZGVsLCB0b2ssIHJlZl90ZXh0cywgbW9kdWxlcywgZGV2aWNlLAogICAgICAgICAgICAgICAgICAgICAgICAgICAgICAgICAgICAgICAgICBtYXhfbGVuPW1heF9sZW4sIGJhdGNoX3NpemU9YXJncy5iYXRjaF9zaXplLAogICAgICAgICAgICAgICAgICAgICAgICAgICAgICAgICAgICAgICAgICBjZW50ZXI9Y2VudGVyKQogICAgdF9yZWYgPSB0aW1lLnBlcmZfY291bnRlcigpIC0gdDAKICAgIHByaW50KGYicmVmZXJlbmNlIHBhc3Mge3RfcmVmOi4xZn1zIikKCiAgICB0MSA9IHRpbWUucGVyZl9jb3VudGVyKCkKICAgIGNvdl9kID0gZHJpZnRfbW9kLmNvbGxlY3RfY292YXJpYW5jZXMobW9kZWwsIHRvaywgZG9tX3RleHRzLCBtb2R1bGVzLCBkZXZpY2UsCiAgICAgICAgICAgICAgICAgICAgICAgICAgICAgICAgICAgICAgICAgIG1heF9sZW49bWF4X2xlbiwgYmF0Y2hfc2l6ZT1hcmdzLmJhdGNoX3NpemUsCiAgICAgICAgICAgICAgICAgICAgICAgICAgICAgICAgICAgICAgICAgIGNlbnRlcj1jZW50ZXIpCiAgICB0X2RvbSA9IHRpbWUucGVyZl9jb3VudGVyKCkgLSB0MQogICAgcHJpbnQoZiJkb21haW4gcGFzcyB7dF9kb206LjFmfXMiKQoKICAgIGRlbCBtb2RlbAogICAgaWYgZGV2aWNlLnR5cGUgPT0gImN1ZGEiOgogICAgICAgIHRvcmNoLmN1ZGEuZW1wdHlfY2FjaGUoKQoKICAgIHRhdXMgPSBbZmxvYXQoeCkgZm9yIHggaW4gYXJncy50YXVzLnNwbGl0KCIsIildCiAgICBzdG9yZSA9IHsibWV0YSI6IHsibW9kZWwiOiBhcmdzLm1vZGVsLCAidGFzayI6IGFyZ3MudGFzaywgIm5fcmVmIjogYXJncy5uX3JlZiwKICAgICAgICAgICAgICAgICAgICAgICJuX2RvbSI6IGFyZ3Mubl9kb20sICJtYXhfbGVuIjogbWF4X2xlbiwgInJfbWF4IjogYXJncy5yX21heCwKICAgICAgICAgICAgICAgICAgICAgICJjZW50ZXJlZCI6IGNlbnRlciwgInJlZiI6IGFyZ3MucmVmLAogICAgICAgICAgICAgICAgICAgICAgIm5fcmVmX2FjdHVhbCI6IGxlbihyZWZfdGV4dHMpLCAibl9kb21fYWN0dWFsIjogbGVuKGRvbV90ZXh0cyksCiAgICAgICAgICAgICAgICAgICAgICAidF9yZWZfcyI6IHRfcmVmLCAidF9kb21fcyI6IHRfZG9tLAogICAgICAgICAgICAgICAgICAgICAgImNvc3RzIjoge246IGRyaWZ0X21vZC5tb2R1bGVfY29zdChtKSBmb3IgbiwgbSBpbiBtb2R1bGVzLml0ZW1zKCl9LAogICAgICAgICAgICAgICAgICAgICAgImRpbXMiOiB7bjogW20uaW5fZmVhdHVyZXMsIG0ub3V0X2ZlYXR1cmVzXSBmb3IgbiwgbSBpbiBtb2R1bGVzLml0ZW1zKCl9fSwKICAgICAgICAgICAgICJ0YXVzIjoge319CiAgICBpZiBhcmdzLmdldjoKICAgICAgICBzdG9yZVsiZ2V2Il0gPSB7fQogICAgICAgIHN0b3JlWyJtZXRhIl1bImdldl9zaHJpbmsiXSA9IGFyZ3MuZ2V2X3NocmluawoKICAgIHQyID0gdGltZS5wZXJmX2NvdW50ZXIoKQogICAgZm9yIHRhdSBpbiB0YXVzOgogICAgICAgIHN0b3JlWyJ0YXVzIl1bc3RyKHRhdSldID0ge30KICAgICMgTW9kdWxlLW1ham9yOiB0aGUgcmVmZXJlbmNlIGVpZ2VuZGVjb21wb3NpdGlvbiBpcyB0aGUgZXhwZW5zaXZlIHN0ZXAgYW5kIGRvZXMKICAgICMgbm90IGRlcGVuZCBvbiB0YXUsIHNvIGl0IGlzIGNvbXB1dGVkIG9uY2UgYW5kIHJldXNlZCBmb3IgZXZlcnkgdGF1LgogICAgZm9yIG5hbWUgaW4gbW9kdWxlczoKICAgICAgICAjIHByb21vdGUgb25lIG1vZHVsZSBhdCBhIHRpbWU7IHRoZSBjYWNoZWQgbWF0cmljZXMgc3RheSBmbG9hdDMyCiAgICAgICAgc2cgPSBjb3ZfZ1tuYW1lXVswXS5kb3VibGUoKQogICAgICAgIHNkID0gY292X2RbbmFtZV1bMF0uZG91YmxlKCkKICAgICAgICBpZiBhcmdzLmdldjoKICAgICAgICAgICAgbXUsIHYsIGVuZXJneSA9IGRyaWZ0X21vZC5nZXZfYmFzaXMoc2QsIHNnLCBzaHJpbms9YXJncy5nZXZfc2hyaW5rLAogICAgICAgICAgICAgICAgICAgICAgICAgICAgICAgICAgICAgICAgICAgICAgICByX2tlZXA9YXJncy5yX21heCkKICAgICAgICAgICAgc3RvcmVbImdldiJdW25hbWVdID0geyJldmFscyI6IG11LmZsb2F0KCkubnVtcHkoKSwKICAgICAgICAgICAgICAgICAgICAgICAgICAgICAgICAgICJiYXNpcyI6IHYuZmxvYXQoKS5udW1weSgpLAogICAgICAgICAgICAgICAgICAgICAgICAgICAgICAgICAgImVuZXJneSI6IGVuZXJneS5mbG9hdCgpLm51bXB5KCksCiAgICAgICAgICAgICAgICAgICAgICAgICAgICAgICAgICAidHJhY2VfZCI6IGZsb2F0KHRvcmNoLmRpYWdvbmFsKHNkKS5zdW0oKSl9CiAgICAgICAgZXZhbHNfZywgZXZlY3NfZyA9IGRyaWZ0X21vZC5yZWZlcmVuY2VfZWlnaChzZykKICAgICAgICBkZWwgc2cKICAgICAgICBmb3IgdGF1IGluIHRhdXM6CiAgICAgICAgICAgIGlmIHRhdSA8PSAwOgogICAgICAgICAgICAgICAgdl9jb21wLCBrID0gTm9uZSwgMCAgICAgICAgICAjIG5vIGRlZmxhdGlvbjogcGxhaW4gdGFyZ2V0IFBDQQogICAgICAgICAgICBlbHNlOgogICAgICAgICAgICAgICAgXywgayA9IGRyaWZ0X21vZC5zdWJzcGFjZV9mcm9tX2VpZ2goZXZhbHNfZywgZXZlY3NfZywgdGF1PXRhdSkKICAgICAgICAgICAgICAgIHZfY29tcCA9IGV2ZWNzX2dbOiwgazpdICAgICAgIyB0cmFpbGluZyBlaWdlbnZlY3RvcnMgc3BhbiAoSS1QKQogICAgICAgICAgICBldmFscywgZXZlY3MsIHRyX2QsIHRyX2RyaWZ0ID0gZHJpZnRfbW9kLmRyaWZ0X3NwZWN0cnVtKAogICAgICAgICAgICAgICAgc2QsIHZfY29tcCwgcl9rZWVwPWFyZ3Mucl9tYXgsIGRldmljZT1kZXZpY2UpCiAgICAgICAgICAgIHN0b3JlWyJ0YXVzIl1bc3RyKHRhdSldW25hbWVdID0gewogICAgICAgICAgICAgICAgImV2YWxzIjogZXZhbHMuZmxvYXQoKS5udW1weSgpLAogICAgICAgICAgICAgICAgImJhc2lzIjogZXZlY3MuZmxvYXQoKS5udW1weSgpLAogICAgICAgICAgICAgICAgInRyYWNlX2QiOiB0cl9kLAogICAgICAgICAgICAgICAgInRyYWNlX2RyaWZ0IjogdHJfZHJpZnQsCiAgICAgICAgICAgICAgICAiayI6IGssCiAgICAgICAgICAgIH0KICAgICAgICBkZWwgZXZhbHNfZywgZXZlY3NfZywgc2QKICAgICAgICBjb3ZfZ1tuYW1lXSA9IE5vbmUKICAgICAgICBjb3ZfZFtuYW1lXSA9IE5vbmUKICAgIGZvciB0YXUgaW4gdGF1czoKICAgICAgICBwZXJfbW9kID0gc3RvcmVbInRhdXMiXVtzdHIodGF1KV0KICAgICAgICBtZWFuX3JhdGlvID0gc3VtKHBlcl9tb2Rbbl1bInRyYWNlX2RyaWZ0Il0gLyBtYXgocGVyX21vZFtuXVsidHJhY2VfZCJdLCAxZS0xMikKICAgICAgICAgICAgICAgICAgICAgICAgIGZvciBuIGluIG1vZHVsZXMpIC8gbGVuKG1vZHVsZXMpCiAgICAgICAgcHJpbnQoZiJ0YXU9e3RhdX06IG1lYW4gZHJpZnQgcmF0aW8ge21lYW5fcmF0aW86LjRmfSB8ICIKICAgICAgICAgICAgICBmIm1lYW4gayB7c3VtKHBlcl9tb2Rbbl1bJ2snXSBmb3IgbiBpbiBtb2R1bGVzKS9sZW4obW9kdWxlcyk6LjFmfSIpCiAgICB0X2VpZyA9IHRpbWUucGVyZl9jb3VudGVyKCkgLSB0MgogICAgc3RvcmVbIm1ldGEiXVsidF9laWdfcyJdID0gdF9laWcKICAgIHByaW50KGYic3BlY3RyYWwgYW5hbHlzaXMge3RfZWlnOi4xZn1zIikKCiAgICB0b3JjaC5zYXZlKHN0b3JlLCBvdXRfcGF0aCkKICAgIHByaW50KCJzYXZlZCIsIG91dF9wYXRoLCBmIih7b3MucGF0aC5nZXRzaXplKG91dF9wYXRoKS8xZTY6LjFmfSBNQikiKQoKICAgIGRpbXMgPSBzdG9yZVsibWV0YSJdWyJkaW1zIl0KICAgIHN1bW0gPSB7ImtleSI6IGtleSwgInRfcmVmX3MiOiB0X3JlZiwgInRfZG9tX3MiOiB0X2RvbSwgInRfZWlnX3MiOiB0X2VpZywKICAgICAgICAgICAgIm5fbW9kdWxlcyI6IGxlbihtb2R1bGVzKSwgInRhdXMiOiB0YXVzLCAiY2VudGVyZWQiOiBjZW50ZXIsCiAgICAgICAgICAgICJyZWYiOiBhcmdzLnJlZiwgIm5fcmVmX2FjdHVhbCI6IGxlbihyZWZfdGV4dHMpLAogICAgICAgICAgICAibl9kb21fYWN0dWFsIjogbGVuKGRvbV90ZXh0cyksCiAgICAgICAgICAgICMgbWVhbiBkcmlmdCByYXRpbyBwZXIgdGF1LCBmb3IgdGhlIHBhcGVyIHRleHQKICAgICAgICAgICAgImRyaWZ0X3JhdGlvIjoge3N0cih0KTogc3VtKHN0b3JlWyJ0YXVzIl1bc3RyKHQpXVtuXVsidHJhY2VfZHJpZnQiXQogICAgICAgICAgICAgICAgICAgICAgICAgICAgICAgICAgICAgICAgLyBtYXgoc3RvcmVbInRhdXMiXVtzdHIodCldW25dWyJ0cmFjZV9kIl0sIDFlLTEyKQogICAgICAgICAgICAgICAgICAgICAgICAgICAgICAgICAgICAgICAgZm9yIG4gaW4gbW9kdWxlcykgLyBsZW4obW9kdWxlcykKICAgICAgICAgICAgICAgICAgICAgICAgICAgIGZvciB0IGluIHRhdXN9LAogICAgICAgICAgICAjIHJlZmVyZW5jZS1zdWJzcGFjZSBkaW1lbnNpb24gayBwZXIgbW9kdWxlLCBhbmQgd2hlcmUgZWFjaCBtb2R1bGUncwogICAgICAgICAgICAjIGxlYWRpbmcgaW5pdGlhbCBkaXJlY3Rpb24gcG9pbnRzLCBzbyB0aGUgcmVmZXJlbmNlIGNvbnRyb2xzIGNhbiBiZQogICAgICAgICAgICAjIHJlYWQgd2l0aG91dCBkb3dubG9hZGluZyB0aGUgcHJvZmlsZSB0ZW5zb3JzCiAgICAgICAgICAgICJrIjoge3N0cih0KToge246IHN0b3JlWyJ0YXVzIl1bc3RyKHQpXVtuXVsiayJdIGZvciBuIGluIG1vZHVsZXN9CiAgICAgICAgICAgICAgICAgIGZvciB0IGluIHRhdXN9LAogICAgICAgICAgICAibGVhZCI6IHtzdHIodCk6IHtuOiBsZWFkX3N1bW1hcnkoc3RvcmVbInRhdXMiXVtzdHIodCldW25dWyJiYXNpcyJdLAogICAgICAgICAgICAgICAgICAgICAgICAgICAgICAgICAgICAgICAgICAgICAgc3RvcmVbInRhdXMiXVtzdHIodCldW25dWyJldmFscyJdLAogICAgICAgICAgICAgICAgICAgICAgICAgICAgICAgICAgICAgICAgICAgICAgc3RvcmVbInRhdXMiXVtzdHIodCldW25dWyJ0cmFjZV9kIl0sCiAgICAgICAgICAgICAgICAgICAgICAgICAgICAgICAgICAgICAgICAgICAgICBkaW1zW25dWzBdKSBmb3IgbiBpbiBtb2R1bGVzfQogICAgICAgICAgICAgICAgICAgICBmb3IgdCBpbiB0YXVzfX0KICAgIGlmIGFyZ3MuZ2V2OgogICAgICAgIHN1bW1bImxlYWQiXVsiZ2V2Il0gPSB7bjogbGVhZF9zdW1tYXJ5KHN0b3JlWyJnZXYiXVtuXVsiYmFzaXMiXSwKICAgICAgICAgICAgICAgICAgICAgICAgICAgICAgICAgICAgICAgICAgICAgICBzdG9yZVsiZ2V2Il1bbl1bImVuZXJneSJdLAogICAgICAgICAgICAgICAgICAgICAgICAgICAgICAgICAgICAgICAgICAgICAgIHN0b3JlWyJnZXYiXVtuXVsidHJhY2VfZCJdLCBkaW1zW25dWzBdKQogICAgICAgICAgICAgICAgICAgICAgICAgICAgICAgZm9yIG4gaW4gbW9kdWxlc30KICAgIHdpdGggb3Blbihvcy5wYXRoLmpvaW4oUFJPRklMRV9ESVIsIGtleSArICIuanNvbiIpLCAidyIpIGFzIGY6CiAgICAgICAganNvbi5kdW1wKHN1bW0sIGYsIGluZGVudD0yKQoKCmlmIF9fbmFtZV9fID09ICJfX21haW5fXyI6CiAgICBtYWluKCkK", "run.py": "IiIiU2luZ2xlLWV4cGVyaW1lbnQgcnVubmVyLgoKICAgIHB5dGhvbiBzcmMvcnVuLnB5IC0tbW9kZWwgcm9iZXJ0YS1iYXNlIC0tdGFzayBjaGVtcHJvdCAtLW1ldGhvZCBkcmlmdCAtLXNlZWQgMQoiIiIKaW1wb3J0IGpzb24KaW1wb3J0IG9zCmltcG9ydCBzeXMKaW1wb3J0IHRpbWUKCmltcG9ydCB0b3JjaAoKc3lzLnBhdGguaW5zZXJ0KDAsIG9zLnBhdGguZGlybmFtZShvcy5wYXRoLmFic3BhdGgoX19maWxlX18pKSkKaW1wb3J0IGNvbW1vbiAgICAgICAgICAgICAgICMgbm9xYTogRTQwMgppbXBvcnQgZGF0YSBhcyBkYXRhX21vZCAgICAgIyBub3FhOiBFNDAyCmltcG9ydCBlbmdpbmUgICAgICAgICAgICAgICAjIG5vcWE6IEU0MDIKaW1wb3J0IHByb2ZpbGVfZHJpZnQgICAgICAgICMgbm9xYTogRTQwMgppbXBvcnQgcnVuc3BlYyAgICAgICAgICAgICAgIyBub3FhOiBFNDAyCmZyb20gcnVuc3BlYyBpbXBvcnQgTkVFRFNfUFJPRklMRSwgcnVuX2lkICAgIyBub3FhOiBFNDAyLEY0MDEgIChyZS1leHBvcnRlZCkKClJPT1QgPSBvcy5wYXRoLmRpcm5hbWUob3MucGF0aC5kaXJuYW1lKG9zLnBhdGguYWJzcGF0aChfX2ZpbGVfXykpKQpSRVNVTFRfRElSID0gb3MucGF0aC5qb2luKFJPT1QsICJydW5zIiwgInJlc3VsdHMiKQoKCmRlZiBtYWluKCk6CiAgICBhID0gcnVuc3BlYy5wYXJzZSgpCiAgICBpZiBhLmRldGVybWluaXN0aWM6CiAgICAgICAgIyBtdXN0IGJlIHNldCBiZWZvcmUgdGhlIGZpcnN0IGN1QkxBUyBjYWxsCiAgICAgICAgb3MuZW52aXJvbi5zZXRkZWZhdWx0KCJDVUJMQVNfV09SS1NQQUNFX0NPTkZJRyIsICI6NDA5Njo4IikKICAgICAgICB0b3JjaC51c2VfZGV0ZXJtaW5pc3RpY19hbGdvcml0aG1zKFRydWUsIHdhcm5fb25seT1UcnVlKQogICAgICAgIHRvcmNoLmJhY2tlbmRzLmN1ZG5uLmRldGVybWluaXN0aWMgPSBUcnVlCiAgICAgICAgdG9yY2guYmFja2VuZHMuY3Vkbm4uYmVuY2htYXJrID0gRmFsc2UKCiAgICBvcy5tYWtlZGlycyhSRVNVTFRfRElSLCBleGlzdF9vaz1UcnVlKQogICAgcmlkID0gcnVuX2lkKGEpCiAgICBvdXRfcGF0aCA9IG9zLnBhdGguam9pbihSRVNVTFRfRElSLCByaWQgKyAiLmpzb24iKQogICAgaWYgb3MucGF0aC5leGlzdHMob3V0X3BhdGgpIGFuZCBub3QgYS5mb3JjZToKICAgICAgICBwcmludCgiU0tJUCAoZXhpc3RzKToiLCByaWQpCiAgICAgICAgcmV0dXJuCgogICAgZGV2aWNlID0gdG9yY2guZGV2aWNlKCJjdWRhIiBpZiB0b3JjaC5jdWRhLmlzX2F2YWlsYWJsZSgpIGVsc2UgImNwdSIpCiAgICBjb21tb24uc2V0X3NlZWQoYS5zZWVkKQoKICAgIHRhc2sgPSBkYXRhX21vZC5sb2FkX3Rhc2soYS50YXNrLCBtYXhfdHJhaW49YS5tYXhfdHJhaW4sIHNlZWQ9MCkKICAgIG1heF9sZW4gPSBhLm1heF9sZW4gb3IgZGF0YV9tb2QuVEFTS19NQVhMRU4uZ2V0KGEudGFzaywgMTI4KQoKICAgIHByb2ZpbGUgPSBOb25lCiAgICBpZiBhLm1ldGhvZCBpbiBORUVEU19QUk9GSUxFOgogICAgICAgIGtleSA9IHByb2ZpbGVfZHJpZnQucHJvZmlsZV9rZXkoYS5tb2RlbCwgYS50YXNrLCBhLm5fcmVmLCBhLm5fZG9tLAogICAgICAgICAgICAgICAgICAgICAgICAgICAgICAgICAgICAgICAgY2VudGVyPShhLmNvdiA9PSAiY2VudGVyZWQiKSwgcmVmPWEucmVmLAogICAgICAgICAgICAgICAgICAgICAgICAgICAgICAgICAgICAgICAgZ2V2PShhLm1ldGhvZCA9PSAiZ2V2IikpCiAgICAgICAgcCA9IG9zLnBhdGguam9pbihwcm9maWxlX2RyaWZ0LlBST0ZJTEVfRElSLCBrZXkgKyAiLnB0IikKICAgICAgICBpZiBub3Qgb3MucGF0aC5leGlzdHMocCk6CiAgICAgICAgICAgIHJhaXNlIFN5c3RlbUV4aXQoIm1pc3NpbmcgcHJvZmlsZTogIiArIHAgKyAiXG5ydW4gc3JjL3Byb2ZpbGVfZHJpZnQucHkgZmlyc3QiKQogICAgICAgIHByb2ZpbGUgPSB0b3JjaC5sb2FkKHAsIHdlaWdodHNfb25seT1GYWxzZSkKCiAgICBtb2RlbCwgdG9rID0gZW5naW5lLmJ1aWxkX21vZGVsKGEubW9kZWwsIHRhc2tbIm51bV9sYWJlbHMiXSwgdGFza1sibXVsdGlsYWJlbCJdLAogICAgICAgICAgICAgICAgICAgICAgICAgICAgICAgICAgICBoZWFkPWEuaGVhZCkKICAgIGluZm8gPSBlbmdpbmUuYXBwbHlfbWV0aG9kKG1vZGVsLCBhLm1ldGhvZCwgYnVkZ2V0X3Jhbms9YS5idWRnZXRfcmFuaywKICAgICAgICAgICAgICAgICAgICAgICAgICAgICAgIGFscGhhPWEuYWxwaGEsIGRyb3BvdXQ9YS5kcm9wb3V0LCBwcm9maWxlPXByb2ZpbGUsCiAgICAgICAgICAgICAgICAgICAgICAgICAgICAgICB0YXU9YS50YXUsIHNjb3JlX21vZGU9YS5zY29yZV9tb2RlLCByaG89YS5yaG8sCiAgICAgICAgICAgICAgICAgICAgICAgICAgICAgICByX21pbj1hLnJfbWluLCBpbml0X21vZGU9YS5pbml0X21vZGUsCiAgICAgICAgICAgICAgICAgICAgICAgICAgICAgICBhbGxvY19tb2RlPWEuYWxsb2NfbW9kZSwKICAgICAgICAgICAgICAgICAgICAgICAgICAgICAgIHRhcmdldD1hLnRhcmdldCwgc2VlZD1hLnNlZWQsIHNjYWxlPWEuc2NhbGUsCiAgICAgICAgICAgICAgICAgICAgICAgICAgICAgICBzY2FsaW5nPWEuc2NhbGluZykKICAgIG1vZGVsLnRvKGRldmljZSkKICAgIGlmIGRldmljZS50eXBlID09ICJjdWRhIjoKICAgICAgICB0b3JjaC5jdWRhLnJlc2V0X3BlYWtfbWVtb3J5X3N0YXRzKCkKCiAgICB0MCA9IHRpbWUucGVyZl9jb3VudGVyKCkKICAgIHJlcyA9IGVuZ2luZS50cmFpbl9ldmFsKG1vZGVsLCB0b2ssIHRhc2ssIGRldmljZSwgaW5mbywKICAgICAgICAgICAgICAgICAgICAgICAgICAgIGVwb2Nocz1hLmVwb2NocywgbHI9YS5sciwgYmF0Y2hfc2l6ZT1hLmJhdGNoX3NpemUsCiAgICAgICAgICAgICAgICAgICAgICAgICAgICBldmFsX2JhdGNoX3NpemU9YS5ldmFsX2JhdGNoX3NpemUsIG1heF9sZW49bWF4X2xlbiwKICAgICAgICAgICAgICAgICAgICAgICAgICAgIHNlZWQ9YS5zZWVkLCB3ZWlnaHRfZGVjYXk9YS53ZWlnaHRfZGVjYXksCiAgICAgICAgICAgICAgICAgICAgICAgICAgICBsb2dfZXZlcnk9MSBpZiBhLnZlcmJvc2UgZWxzZSAwLCBhbXA9YS5hbXAsCiAgICAgICAgICAgICAgICAgICAgICAgICAgICBzYXZlX3ByZWRzPW5vdCBhLm5vX3NhdmVfcHJlZHMpCiAgICB3YWxsID0gdGltZS5wZXJmX2NvdW50ZXIoKSAtIHQwCgogICAgcmFua3MgPSBpbmZvLmdldCgicmFua3MiLCB7fSkKICAgIHJlY29yZCA9IHsKICAgICAgICAiaWQiOiByaWQsICJhcmdzIjogdmFycyhhKSwgIndhbGxfcyI6IHdhbGwsCiAgICAgICAgIm5fdHJhaW4iOiBsZW4odGFza1sic3BsaXRzIl1bInRyYWluIl1bMF0pLAogICAgICAgICJuX2RldiI6IGxlbih0YXNrWyJzcGxpdHMiXVsiZGV2Il1bMF0pLAogICAgICAgICJuX3Rlc3QiOiBsZW4odGFza1sic3BsaXRzIl1bInRlc3QiXVswXSksCiAgICAgICAgIm51bV9sYWJlbHMiOiB0YXNrWyJudW1fbGFiZWxzIl0sICJtZXRyaWMiOiB0YXNrWyJtZXRyaWMiXSwKICAgICAgICAibWF4X2xlbiI6IG1heF9sZW4sCiAgICAgICAgImJ1ZGdldCI6IHtrOiBpbmZvW2tdIGZvciBrIGluICgiYnVkZ2V0X3JhbmsiLCAiYnVkZ2V0X3BhcmFtcyIsICJuX21vZHVsZXMiKQogICAgICAgICAgICAgICAgICAgaWYgayBpbiBpbmZvfSwKICAgICAgICAic3BlbnRfcGFyYW1zIjogaW5mby5nZXQoInNwZW50X3BhcmFtcyIpLAogICAgICAgICJyYW5rX2hpc3QiOiB7c3RyKGspOiBzdW0oMSBmb3IgdiBpbiByYW5rcy52YWx1ZXMoKSBpZiB2ID09IGspCiAgICAgICAgICAgICAgICAgICAgICBmb3IgayBpbiBzb3J0ZWQoc2V0KHJhbmtzLnZhbHVlcygpKSl9IGlmIHJhbmtzIGVsc2Uge30sCiAgICAgICAgInJhbmtzIjoge2s6IGludCh2KSBmb3IgaywgdiBpbiByYW5rcy5pdGVtcygpfSwKICAgICAgICAidmVyc2lvbnMiOiB7InRvcmNoIjogdG9yY2guX192ZXJzaW9uX18sCiAgICAgICAgICAgICAgICAgICAgICJ0cmFuc2Zvcm1lcnMiOiBfX2ltcG9ydF9fKCJ0cmFuc2Zvcm1lcnMiKS5fX3ZlcnNpb25fX30sCiAgICAgICAgInJlc3VsdCI6IHJlcywKICAgIH0KICAgIGNvbW1vbi5zYXZlX2pzb24ocmVjb3JkLCBvdXRfcGF0aCkKICAgIG0gPSB0YXNrWyJtZXRyaWMiXQogICAgcHJpbnQoZiJET05FIHtyaWR9XG4gIHRlc3Qge219PXtyZXNbJ3Rlc3QnXVttXTouNGZ9ICIKICAgICAgICAgIGYiZGV2PXtyZXNbJ2Rldl9iZXN0J10uZ2V0KG0sIGZsb2F0KCduYW4nKSk6LjRmfSAiCiAgICAgICAgICBmImFkYXB0ZXJfcGFyYW1zPXtyZXNbJ3BhcmFtc19hZGFwdGVyJ119ICIKICAgICAgICAgIGYidGltZT17cmVzWyd0cmFpbl90aW1lX3MnXTouMGZ9cyIpCgoKaWYgX19uYW1lX18gPT0gIl9fbWFpbl9fIjoKICAgIG1haW4oKQo=", "runspec.py": "IiIiQ29tbWFuZC1saW5lIHNwZWMgb2Ygb25lIHJ1biBhbmQgaXRzIHJlc3VsdCBpZC4KCktlcHQgZnJlZSBvZiB0b3JjaC90cmFuc2Zvcm1lcnMgaW1wb3J0cyBzbyB0aGUgZ3JpZCBjYW4gZGVjaWRlIHdoaWNoIHJ1bnMgYXJlCmFscmVhZHkgZmluaXNoZWQgd2l0aG91dCBwYXlpbmcgZm9yIGEgbW9kZWwtbGlicmFyeSBpbXBvcnQgcGVyIGNvbW1hbmQuCiIiIgppbXBvcnQgYXJncGFyc2UKCk5FRURTX1BST0ZJTEUgPSB7ImV2YSIsICJldmFfd2hpdGUiLCAiZHJpZnQiLCAiZHJpZnRfYWJzIiwgImRyaWZ0X25vZGVmbGF0ZSIsICJnZXYifQoKCmRlZiBidWlsZF9wYXJzZXIoKToKICAgIGFwID0gYXJncGFyc2UuQXJndW1lbnRQYXJzZXIoKQogICAgYXAuYWRkX2FyZ3VtZW50KCItLW1vZGVsIiwgZGVmYXVsdD0icm9iZXJ0YS1iYXNlIikKICAgIGFwLmFkZF9hcmd1bWVudCgiLS10YXNrIiwgZGVmYXVsdD0iY2hlbXByb3QiKQogICAgYXAuYWRkX2FyZ3VtZW50KCItLW1ldGhvZCIsIGRlZmF1bHQ9ImxvcmEiKQogICAgYXAuYWRkX2FyZ3VtZW50KCItLXNlZWQiLCB0eXBlPWludCwgZGVmYXVsdD0xKQogICAgYXAuYWRkX2FyZ3VtZW50KCItLWJ1ZGdldF9yYW5rIiwgdHlwZT1pbnQsIGRlZmF1bHQ9OCkKICAgIGFwLmFkZF9hcmd1bWVudCgiLS1hbHBoYSIsIHR5cGU9ZmxvYXQsIGRlZmF1bHQ9MTYuMCkKICAgIGFwLmFkZF9hcmd1bWVudCgiLS1kcm9wb3V0IiwgdHlwZT1mbG9hdCwgZGVmYXVsdD0wLjApCiAgICBhcC5hZGRfYXJndW1lbnQoIi0tbHIiLCB0eXBlPWZsb2F0LCBkZWZhdWx0PTNlLTQpCiAgICBhcC5hZGRfYXJndW1lbnQoIi0tZXBvY2hzIiwgdHlwZT1pbnQsIGRlZmF1bHQ9MTApCiAgICBhcC5hZGRfYXJndW1lbnQoIi0tYmF0Y2hfc2l6ZSIsIHR5cGU9aW50LCBkZWZhdWx0PTMyKQogICAgYXAuYWRkX2FyZ3VtZW50KCItLWV2YWxfYmF0Y2hfc2l6ZSIsIHR5cGU9aW50LCBkZWZhdWx0PTY0KQogICAgYXAuYWRkX2FyZ3VtZW50KCItLW1heF9sZW4iLCB0eXBlPWludCwgZGVmYXVsdD1Ob25lKQogICAgYXAuYWRkX2FyZ3VtZW50KCItLW1heF90cmFpbiIsIHR5cGU9aW50LCBkZWZhdWx0PU5vbmUpCiAgICBhcC5hZGRfYXJndW1lbnQoIi0td2VpZ2h0X2RlY2F5IiwgdHlwZT1mbG9hdCwgZGVmYXVsdD0wLjAxKQogICAgYXAuYWRkX2FyZ3VtZW50KCItLXRhdSIsIHR5cGU9ZmxvYXQsIGRlZmF1bHQ9MC45NSkKICAgIGFwLmFkZF9hcmd1bWVudCgiLS1yaG8iLCB0eXBlPWZsb2F0LCBkZWZhdWx0PTIuMCkKICAgIGFwLmFkZF9hcmd1bWVudCgiLS1zY29yZV9tb2RlIiwgZGVmYXVsdD0icmVsYXRpdmUiKQogICAgYXAuYWRkX2FyZ3VtZW50KCItLWluaXRfbW9kZSIsIGRlZmF1bHQ9ImRyaWZ0IikKICAgIGFwLmFkZF9hcmd1bWVudCgiLS1hbGxvY19tb2RlIiwgZGVmYXVsdD0iZHJpZnQiKQogICAgYXAuYWRkX2FyZ3VtZW50KCItLXJfbWluIiwgdHlwZT1pbnQsIGRlZmF1bHQ9MCkKICAgIGFwLmFkZF9hcmd1bWVudCgiLS10YXJnZXQiLCBkZWZhdWx0PSJhbGwiKQogICAgYXAuYWRkX2FyZ3VtZW50KCItLWNvdiIsIGRlZmF1bHQ9ImNlbnRlcmVkIiwgY2hvaWNlcz1bImNlbnRlcmVkIiwgInJhdyJdLAogICAgICAgICAgICAgICAgICAgIGhlbHA9InByb2ZpbGUgdHlwZSBmb3IgRVZBL0RSSUZUOiBjb3ZhcmlhbmNlIChhcyBpbiBFVkEncyAiCiAgICAgICAgICAgICAgICAgICAgICAgICAicmVmZXJlbmNlIGltcGxlbWVudGF0aW9uKSBvciByYXcgc2Vjb25kIG1vbWVudCIpCiAgICBhcC5hZGRfYXJndW1lbnQoIi0tc2NhbGUiLCBkZWZhdWx0PSJhZGp1c3RlZCIsIGNob2ljZXM9WyJhZGp1c3RlZCIsICJyYW5rIl0sCiAgICAgICAgICAgICAgICAgICAgaGVscD0iYWRhcHRlciBzY2FsaW5nIGZvciByYW5rLXJlZGlzdHJpYnV0aW5nIG1ldGhvZHM6ICIKICAgICAgICAgICAgICAgICAgICAgICAgICInYWRqdXN0ZWQnIGtlZXBzIGFscGhhL3Igb2YgdGhlIHVuaWZvcm0gYnVkZ2V0IHJhbmsgZm9yICIKICAgICAgICAgICAgICAgICAgICAgICAgICJldmVyeSBtb2R1bGUgKEVWQSdzIGRlZmF1bHQpLCAncmFuaycgdXNlcyBhbHBoYS9yX20iKQogICAgYXAuYWRkX2FyZ3VtZW50KCItLW5fcmVmIiwgdHlwZT1pbnQsIGRlZmF1bHQ9MTAyNCkKICAgIGFwLmFkZF9hcmd1bWVudCgiLS1uX2RvbSIsIHR5cGU9aW50LCBkZWZhdWx0PTEwMjQpCiAgICBhcC5hZGRfYXJndW1lbnQoIi0tcmVmIiwgZGVmYXVsdD0id2lraXRleHQiLAogICAgICAgICAgICAgICAgICAgIGhlbHA9InJlZmVyZW5jZSBjb3JwdXMgb2YgdGhlIHByb2ZpbGUgKHNlZSBkYXRhLlJFRkVSRU5DRV9LSU5EUykiKQogICAgYXAuYWRkX2FyZ3VtZW50KCItLXRhZyIsIGRlZmF1bHQ9IiIpCiAgICBhcC5hZGRfYXJndW1lbnQoIi0tYW1wIiwgYWN0aW9uPSJzdG9yZV90cnVlIiwKICAgICAgICAgICAgICAgICAgICBoZWxwPSJmcDE2IGF1dG9jYXN0OyB1c2Ugb24gVHVyaW5nKyBHUFVzLCBOT1Qgb24gUGFzY2FsICIKICAgICAgICAgICAgICAgICAgICAgICAgICIoR1AxMHggcnVucyBmcDE2IGF0IDEvNjQgcmF0ZSkiKQogICAgYXAuYWRkX2FyZ3VtZW50KCItLWRldGVybWluaXN0aWMiLCBhY3Rpb249InN0b3JlX3RydWUiLAogICAgICAgICAgICAgICAgICAgIGhlbHA9ImRldGVybWluaXN0aWMga2VybmVscyAoZm9yIGZwMzIgcmVydW5zOiB0d28gcnVucyB3aXRoIHRoZSAiCiAgICAgICAgICAgICAgICAgICAgICAgICAic2FtZSBzZWVkIHRoZW4gZ2l2ZSB0aGUgc2FtZSByZXN1bHQpIikKICAgIGFwLmFkZF9hcmd1bWVudCgiLS1oZWFkIiwgZGVmYXVsdD0iZGVmYXVsdCIsIGNob2ljZXM9WyJkZWZhdWx0IiwgImxpbmVhciJdLAogICAgICAgICAgICAgICAgICAgIGhlbHA9ImNsYXNzaWZpY2F0aW9uIGhlYWQ6IHRoZSBiYWNrYm9uZSdzIG93biAoUm9CRVJUYTogZGVuc2UsICIKICAgICAgICAgICAgICAgICAgICAgICAgICJ0YW5oLCBsaW5lYXIpIG9yIGEgc2luZ2xlIGxpbmVhciBsYXllciIpCiAgICBhcC5hZGRfYXJndW1lbnQoIi0tc2NhbGluZyIsIGRlZmF1bHQ9ImFscGhhX3IiLCBjaG9pY2VzPVsiYWxwaGFfciIsICJyc2xvcmEiXSwKICAgICAgICAgICAgICAgICAgICBoZWxwPSJMb1JBIHNjYWxlIGZvciB1bmlmb3JtLXJhbmsgbWV0aG9kczogYWxwaGEvciwgb3IgcnNMb1JBJ3MgIgogICAgICAgICAgICAgICAgICAgICAgICAgImFscGhhL3NxcnQocikiKQogICAgYXAuYWRkX2FyZ3VtZW50KCItLW5vX3NhdmVfcHJlZHMiLCBhY3Rpb249InN0b3JlX3RydWUiLAogICAgICAgICAgICAgICAgICAgIGhlbHA9ImRvIG5vdCBzdG9yZSBwZXItZXhhbXBsZSB0ZXN0IHByZWRpY3Rpb25zIikKICAgIGFwLmFkZF9hcmd1bWVudCgiLS1mb3JjZSIsIGFjdGlvbj0ic3RvcmVfdHJ1ZSIpCiAgICBhcC5hZGRfYXJndW1lbnQoIi0tdmVyYm9zZSIsIGFjdGlvbj0ic3RvcmVfdHJ1ZSIpCiAgICByZXR1cm4gYXAKCgpkZWYgZmluYWxpemUoYSk6CiAgICAiIiJTZXR0aW5ncyBpbXBsaWVkIGJ5IG90aGVycy4gVGhlIEdFViBpbml0aWFsaXNhdGlvbiBpcyBjb21wYXJlZCBhdCB1bmlmb3JtCiAgICByYW5rLCBzbyBpdHMgaWQgcmVjb3JkcyB0aGF0IGV4cGxpY2l0bHkuIiIiCiAgICBpZiBhLm1ldGhvZCA9PSAiZ2V2IjoKICAgICAgICBhLmluaXRfbW9kZSwgYS5hbGxvY19tb2RlID0gImdldiIsICJ1bmlmb3JtIgogICAgcmV0dXJuIGEKCgpkZWYgcGFyc2UoYXJndj1Ob25lKToKICAgIHJldHVybiBmaW5hbGl6ZShidWlsZF9wYXJzZXIoKS5wYXJzZV9hcmdzKGFyZ3YpKQoKCmRlZiBwcm9maWxlX2tleShtb2RlbF9uYW1lLCB0YXNrLCBuX3JlZiwgbl9kb20sIGNlbnRlcj1UcnVlLCByZWY9Indpa2l0ZXh0IiwKICAgICAgICAgICAgICAgIGdldj1GYWxzZSk6CiAgICBzYWZlID0gbW9kZWxfbmFtZS5yZXBsYWNlKCIvIiwgIl9fIikKICAgICMgY2VudHJlZCAoY292YXJpYW5jZSkgcHJvZmlsZXMgYXJlIHRoZSBkZWZhdWx0OyB0aGUgc3VmZml4IGtlZXBzIHRoZW0gZnJvbQogICAgIyBldmVyIGJlaW5nIGNvbmZ1c2VkIHdpdGggcmF3IHNlY29uZC1tb21lbnQgcHJvZmlsZXMgY2FjaGVkIGVhcmxpZXIuIFRoZQogICAgIyBkZWZhdWx0IFdpa2lUZXh0IHJlZmVyZW5jZSBjYXJyaWVzIG5vIHN1ZmZpeCwgc28gZXhpc3Rpbmcga2V5cyBhcmUgdW5jaGFuZ2VkLgogICAgcmV0dXJuIChmIntzYWZlfV9fe3Rhc2t9X19yZWZ7bl9yZWZ9X19kb217bl9kb219IgogICAgICAgICAgICArICgiIiBpZiByZWYgPT0gIndpa2l0ZXh0IiBlbHNlIGYiX197cmVmfSIpCiAgICAgICAgICAgICsgKCJfX2NlbiIgaWYgY2VudGVyIGVsc2UgIiIpICsgKCJfX2dldiIgaWYgZ2V2IGVsc2UgIiIpKQoKCmRlZiBydW5faWQoYSk6CiAgICBiaXRzID0gW2EubW9kZWwucmVwbGFjZSgiLyIsICJfXyIpLCBhLnRhc2ssIGEubWV0aG9kLCBmInJ7YS5idWRnZXRfcmFua30iLAogICAgICAgICAgICBmImxye2EubHI6Z30iLCBmInN7YS5zZWVkfSJdCiAgICBpZiBhLm1ldGhvZCBpbiBORUVEU19QUk9GSUxFOgogICAgICAgIGJpdHMuYXBwZW5kKGYidGF1e2EudGF1Omd9IikKICAgICAgICBiaXRzLmFwcGVuZChhLnNjb3JlX21vZGUpCiAgICAgICAgYml0cy5hcHBlbmQoZiJpbml0LXthLmluaXRfbW9kZX0iKQogICAgICAgIGJpdHMuYXBwZW5kKGYiYWxsb2Mte2EuYWxsb2NfbW9kZX0iKQogICAgICAgIGJpdHMuYXBwZW5kKGYicmhve2EucmhvOmd9IikKICAgICAgICBiaXRzLmFwcGVuZChmImNvdi17YS5jb3Z9IikKICAgICAgICBiaXRzLmFwcGVuZChmInNjLXthLnNjYWxlfSIpCiAgICAgICAgIyBvbmx5IG5vbi1kZWZhdWx0IHByb2ZpbGVzIGFkZCB0byB0aGUgaWQsIHNvIGV4aXN0aW5nIGlkcyBhcmUgdW5jaGFuZ2VkCiAgICAgICAgaWYgYS5yZWYgIT0gIndpa2l0ZXh0IjoKICAgICAgICAgICAgYml0cy5hcHBlbmQoZiJyZWYte2EucmVmfSIpCiAgICAgICAgaWYgKGEubl9yZWYsIGEubl9kb20pICE9ICgxMDI0LCAxMDI0KToKICAgICAgICAgICAgYml0cy5hcHBlbmQoZiJwcm9me2Eubl9yZWZ9LXthLm5fZG9tfSIpCiAgICBpZiBhLmRldGVybWluaXN0aWM6CiAgICAgICAgYml0cy5hcHBlbmQoImRldCIpCiAgICAjIGxpa2UgdGhlIHByb2ZpbGUgYml0cyBhYm92ZSwgb25seSBub24tZGVmYXVsdCB2YWx1ZXMgYWRkIHRvIHRoZSBpZAogICAgaWYgZ2V0YXR0cihhLCAiaGVhZCIsICJkZWZhdWx0IikgIT0gImRlZmF1bHQiOgogICAgICAgIGJpdHMuYXBwZW5kKGYiaGVhZC17YS5oZWFkfSIpCiAgICBpZiBnZXRhdHRyKGEsICJzY2FsaW5nIiwgImFscGhhX3IiKSAhPSAiYWxwaGFfciI6CiAgICAgICAgYml0cy5hcHBlbmQoZiJzY2FsLXthLnNjYWxpbmd9IikKICAgIGlmIGEudGFyZ2V0ICE9ICJhbGwiOgogICAgICAgIGJpdHMuYXBwZW5kKCJ0Z3QtIiArIGEudGFyZ2V0KQogICAgaWYgYS5tYXhfdHJhaW46CiAgICAgICAgYml0cy5hcHBlbmQoZiJue2EubWF4X3RyYWlufSIpCiAgICBpZiBhLnRhZzoKICAgICAgICBiaXRzLmFwcGVuZChhLnRhZykKICAgIHJldHVybiAiX18iLmpvaW4oYml0cykK", "grid.py": "...", "analyze.py": "...", "download_data.py": "IiIiRmV0Y2ggZXZlcnkgZGF0YXNldCB1c2VkIGluIHRoZSBwYXBlci4gSWRlbXBvdGVudDsgc2FmZSB0byByZS1ydW4uIiIiCmltcG9ydCBvcwppbXBvcnQgc3lzCmltcG9ydCB0aW1lCmltcG9ydCB1cmxsaWIucmVxdWVzdAoKUk9PVCA9IG9zLnBhdGguZGlybmFtZShvcy5wYXRoLmRpcm5hbWUob3MucGF0aC5hYnNwYXRoKF9fZmlsZV9fKSkpCkRBVEEgPSBvcy5wYXRoLmpvaW4oUk9PVCwgImRhdGEiKQoKUzMgPSAiaHR0cHM6Ly9hbGxlbm5scC5zMy11cy13ZXN0LTIuYW1hem9uYXdzLmNvbS9kb250X3N0b3BfcHJldHJhaW5pbmcvZGF0YSIKSEYgPSAiaHR0cHM6Ly9odWdnaW5nZmFjZS5jby9hcGkvZGF0YXNldHMiCgpGSUxFUyA9IFsKICAgICMgQ2hlbVByb3Q6IDEzLXdheSBjaGVtaWNhbC1wcm90ZWluIHJlbGF0aW9uIGNsYXNzaWZpY2F0aW9uIChCaW9DcmVhdGl2ZSBWSSksCiAgICAjIGluIHRoZSBzcGxpdCByZWxlYXNlZCB3aXRoIEd1cnVyYW5nYW4gZXQgYWwuICgyMDIwKS4KICAgIChmIntTM30vY2hlbXByb3QvdHJhaW4uanNvbmwiLCAiY2hlbXByb3QvdHJhaW4uanNvbmwiKSwKICAgIChmIntTM30vY2hlbXByb3QvZGV2Lmpzb25sIiwgImNoZW1wcm90L2Rldi5qc29ubCIpLAogICAgKGYie1MzfS9jaGVtcHJvdC90ZXN0Lmpzb25sIiwgImNoZW1wcm90L3Rlc3QuanNvbmwiKSwKICAgICMgUkNULTIwazogNS13YXkgc2VudGVuY2Utcm9sZSBjbGFzc2lmaWNhdGlvbiBpbiBSQ1QgYWJzdHJhY3RzLgogICAgKGYie1MzfS9yY3QtMjBrL3RyYWluLmpzb25sIiwgInJjdDIway90cmFpbi5qc29ubCIpLAogICAgKGYie1MzfS9yY3QtMjBrL2Rldi5qc29ubCIsICJyY3QyMGsvZGV2Lmpzb25sIiksCiAgICAoZiJ7UzN9L3JjdC0yMGsvdGVzdC5qc29ubCIsICJyY3QyMGsvdGVzdC5qc29ubCIpLAogICAgIyBIYWxsbWFya3Mgb2YgQ2FuY2VyLCBzZW50ZW5jZS1sZXZlbCByZWxlYXNlIChhZ2dyZWdhdGVkIHRvIGRvY3VtZW50cyBpbiBkYXRhLnB5KS4KICAgIChmIntIRn0vcWFuYXN0ZWsvSG9DL3BhcnF1ZXQvSG9DL3RyYWluLzAucGFycXVldCIsICJob2MvdHJhaW4ucGFycXVldCIpLAogICAgKGYie0hGfS9xYW5hc3Rlay9Ib0MvcGFycXVldC9Ib0MvdmFsaWRhdGlvbi8wLnBhcnF1ZXQiLCAiaG9jL3ZhbGlkYXRpb24ucGFycXVldCIpLAogICAgKGYie0hGfS9xYW5hc3Rlay9Ib0MvcGFycXVldC9Ib0MvdGVzdC8wLnBhcnF1ZXQiLCAiaG9jL3Rlc3QucGFycXVldCIpLAogICAgIyBHZW5lcmFsLWRvbWFpbiByZWZlcmVuY2UgY29ycHVzIGZvciB0aGUgRFJJRlQgY29udHJhc3QuCiAgICAoZiJ7SEZ9L1NhbGVzZm9yY2Uvd2lraXRleHQvcGFycXVldC93aWtpdGV4dC0xMDMtcmF3LXYxL3ZhbGlkYXRpb24vMC5wYXJxdWV0IiwKICAgICAicmVmZXJlbmNlL3dpa2l0ZXh0X3ZhbC5wYXJxdWV0IiksCiAgICAoZiJ7SEZ9L1NhbGVzZm9yY2Uvd2lraXRleHQvcGFycXVldC93aWtpdGV4dC0xMDMtcmF3LXYxL3Rlc3QvMC5wYXJxdWV0IiwKICAgICAicmVmZXJlbmNlL3dpa2l0ZXh0X3Rlc3QucGFycXVldCIpLAogICAgIyBBIHNlY29uZCBnZW5lcmFsLWRvbWFpbiByZWZlcmVuY2UgKG5ld3MpLCBmb3IgdGhlIHJlZmVyZW5jZS1jb3JwdXMgY29udHJvbC4KICAgIChmIntIRn0vYWJpc2VlL2Nubl9kYWlseW1haWwvcGFycXVldC8zLjAuMC90ZXN0LzAucGFycXVldCIsCiAgICAgInJlZmVyZW5jZS9jbm5fZGFpbHltYWlsX3Rlc3QucGFycXVldCIpLAogICAgIyBDbGluaWNhbC1zdHlsZSB0ZXh0OiBNVFNhbXBsZXMgbWVkaWNhbCB0cmFuc2NyaXB0aW9ucyAoc3BlY2lhbHR5IGxhYmVscyksCiAgICAjIHJlZ3JvdXBlZCBpbnRvIGEgc3BlY2lhbHR5LWNsYXNzaWZpY2F0aW9uIHRhc2sgaW4gZGF0YS5weS4KICAgIChmIntIRn0vZ2FsaWxlby1haS9tZWRpY2FsX3RyYW5zY3JpcHRpb25fNDAvcGFycXVldC9kZWZhdWx0L3RyYWluLzAucGFycXVldCIsCiAgICAgIm10c2FtcGxlcy90cmFpbi5wYXJxdWV0IiksCiAgICAoZiJ7SEZ9L2dhbGlsZW8tYWkvbWVkaWNhbF90cmFuc2NyaXB0aW9uXzQwL3BhcnF1ZXQvZGVmYXVsdC90ZXN0LzAucGFycXVldCIsCiAgICAgIm10c2FtcGxlcy90ZXN0LnBhcnF1ZXQiKSwKXQoKCmRlZiBtYWluKCk6CiAgICBvayA9IFRydWUKICAgIGZvciB1cmwsIHJlbCBpbiBGSUxFUzoKICAgICAgICBkc3QgPSBvcy5wYXRoLmpvaW4oREFUQSwgcmVsKQogICAgICAgIG9zLm1ha2VkaXJzKG9zLnBhdGguZGlybmFtZShkc3QpLCBleGlzdF9vaz1UcnVlKQogICAgICAgIGlmIG9zLnBhdGguZXhpc3RzKGRzdCkgYW5kIG9zLnBhdGguZ2V0c2l6ZShkc3QpID4gMDoKICAgICAgICAgICAgcHJpbnQoZiJoYXZlICB7cmVsfSIpCiAgICAgICAgICAgIGNvbnRpbnVlCiAgICAgICAgcHJpbnQoZiJnZXQgICB7cmVsfSAuLi4gIiwgZW5kPSIiLCBmbHVzaD1UcnVlKQogICAgICAgICMgdGhlIEh1YiBpbnRlcm1pdHRlbnRseSBhbnN3ZXJzIDUwMzsgcmV0cnkgd2l0aCBiYWNrb2ZmIGJlZm9yZSBnaXZpbmcgdXAKICAgICAgICBmb3IgYXR0ZW1wdCBpbiByYW5nZSg2KToKICAgICAgICAgICAgdHJ5OgogICAgICAgICAgICAgICAgcmVxID0gdXJsbGliLnJlcXVlc3QuUmVxdWVzdCh1cmwsIGhlYWRlcnM9eyJVc2VyLUFnZW50IjogIk1vemlsbGEvNS4wIn0pCiAgICAgICAgICAgICAgICB3aXRoIHVybGxpYi5yZXF1ZXN0LnVybG9wZW4ocmVxLCB0aW1lb3V0PTE4MCkgYXMgcjoKICAgICAgICAgICAgICAgICAgICBib2R5ID0gci5yZWFkKCkKICAgICAgICAgICAgICAgIHdpdGggb3Blbihkc3QsICJ3YiIpIGFzIGY6CiAgICAgICAgICAgICAgICAgICAgZi53cml0ZShib2R5KQogICAgICAgICAgICAgICAgcHJpbnQoZiJ7b3MucGF0aC5nZXRzaXplKGRzdCkvMWU2Oi4yZn0gTUIiKQogICAgICAgICAgICAgICAgYnJlYWsKICAgICAgICAgICAgZXhjZXB0IEV4Y2VwdGlvbiBhcyBlOiAgICAgICAgICAgICAgICAgICAgICAgICAgICAgICAjIG5vcWE6IEJMRTAwMQogICAgICAgICAgICAgICAgaWYgYXR0ZW1wdCA9PSA1OgogICAgICAgICAgICAgICAgICAgIG9rID0gRmFsc2UKICAgICAgICAgICAgICAgICAgICBwcmludCgiRkFJTEVEOiIsIGUpCiAgICAgICAgICAgICAgICBlbHNlOgogICAgICAgICAgICAgICAgICAgIHdhaXQgPSAxNSAqIDIgKiogYXR0ZW1wdAogICAgICAgICAgICAgICAgICAgIHByaW50KGYicmV0cnkgaW4ge3dhaXR9cyAoe2V9KSAuLi4gIiwgZW5kPSIiLCBmbHVzaD1UcnVlKQogICAgICAgICAgICAgICAgICAgIHRpbWUuc2xlZXAod2FpdCkKICAgIGlmIG5vdCBvazoKICAgICAgICBzeXMuZXhpdCgxKQogICAgcHJpbnQoImFsbCBkYXRhIHByZXNlbnQiKQoKCmlmIF9fbmFtZV9fID09ICJfX21haW5fXyI6CiAgICBtYWluKCkK", "divergence.py": "IiIiVGF1LWZyZWUgZGl2ZXJnZW5jZXMgYmV0d2VlbiB0aGUgdGFyZ2V0IGFuZCByZWZlcmVuY2UgYWN0aXZhdGlvbiBjb3ZhcmlhbmNlcy4KCkZvciBlYWNoIGFkYXB0YWJsZSBtb2R1bGUgb2YgYSBiYWNrYm9uZSwgY29tcGFyZXMgdGhlIGNvdmFyaWFuY2Ugb2YgdGhlIHRhcmdldAp0YXNrJ3MgaW5wdXRzIChTaWdtYV9EKSB3aXRoIHRoYXQgb2YgdGhlIGdlbmVyYWwtZG9tYWluIHJlZmVyZW5jZSAoU2lnbWFfRyksIGFuZApjYWxpYnJhdGVzIHRoZSBudW1iZXJzIGFnYWluc3QgdGhlIGRpdmVyZ2VuY2UgYmV0d2VlbiB0d28gZGlzam9pbnQgaGFsdmVzIG9mIHRoZQpyZWZlcmVuY2UgY29ycHVzIChTaWdtYV9HJyB2cyBTaWdtYV9HOiB0aGUgc2FtcGxpbmctbm9pc2UgZmxvb3IpLiBPbiBDaGVtUHJvdAppdCBhbHNvIHNjb3JlcyB0d28gZnVydGhlciBjb3Jwb3JhIGFnYWluc3QgdGhlIHJlZmVyZW5jZSAobmV3cyB0ZXh0IGFuZAp1bmlmb3JtbHkgcmFuZG9tIHRva2VucykgdG8gcGxhY2UgdGhlIGJpb21lZGljYWwgdGFza3Mgb24gYSBzY2FsZS4KClRocmVlIGRpdmVyZ2VuY2VzLCBlYWNoIG9uIHRyYWNlLW5vcm1hbGlzZWQgY292YXJpYW5jZXMgQyA9IFNpZ21hIC8gdHIoU2lnbWEpLApzbyB0aGF0IG9ubHkgdGhlIGdlb21ldHJ5LCBub3QgdGhlIG92ZXJhbGwgZW5lcmd5LCBpcyBjb21wYXJlZDoKICBDT1JBTCAgICB8fENfQSAtIENfQnx8X0YgLyB8fENfQnx8X0YgICAgICAgICAgIChTdW4gZXQgYWwuLCAyMDE2KQogIEJ1cmVzICAgIGRfQldeMiA9IDIgLSAyIHRyKChDX0JeMS8yIENfQSBDX0JeMS8yKV4xLzIpCiAgSmVmZnJleXMgKHRyKEEnXi0xIEInKSArIHRyKEInXi0xIEEnKSkvKDJkKSAtIDEsIGEgbG9nLWRldCAoR2F1c3NpYW4gS0wpCiAgICAgICAgICAgZGl2ZXJnZW5jZSBwZXIgZGltZW5zaW9uLCB3aXRoIGJvdGggY292YXJpYW5jZXMgc2hydW5rIGJ5IDAuMQogICAgICAgICAgIHRvd2FyZHMgYSBzY2FsZWQgaWRlbnRpdHkgc28gdGhhdCB0aGV5IGFyZSBpbnZlcnRpYmxlLgoKICAgIHB5dGhvbiBzcmMvZGl2ZXJnZW5jZS5weSAtLW1vZGVsIHJvYmVydGEtYmFzZSAtLXRhc2sgY2hlbXByb3QKIiIiCmltcG9ydCBhcmdwYXJzZQppbXBvcnQganNvbgppbXBvcnQgb3MKaW1wb3J0IHN5cwppbXBvcnQgdGltZQoKaW1wb3J0IHRvcmNoCgpzeXMucGF0aC5pbnNlcnQoMCwgb3MucGF0aC5kaXJuYW1lKG9zLnBhdGguYWJzcGF0aChfX2ZpbGVfXykpKQppbXBvcnQgZGF0YSBhcyBkYXRhX21vZCAgICAgIyBub3FhOiBFNDAyCmltcG9ydCBkcmlmdCBhcyBkcmlmdF9tb2QgICAjIG5vcWE6IEU0MDIKClJPT1QgPSBvcy5wYXRoLmRpcm5hbWUob3MucGF0aC5kaXJuYW1lKG9zLnBhdGguYWJzcGF0aChfX2ZpbGVfXykpKQpPVVRfRElSID0gb3MucGF0aC5qb2luKFJPT1QsICJydW5zIiwgImRpdmVyZ2VuY2UiKQoKCmRlZiBfbm9ybShzKToKICAgIHMgPSAwLjUgKiAocyArIHMuVCkKICAgIHJldHVybiBzIC8gdG9yY2guZGlhZ29uYWwocykuc3VtKCkuY2xhbXBfbWluKDFlLTMwKQoKCmRlZiBfc2hyaW5rKGMsIGc9MC4xKToKICAgIGQgPSBjLnNoYXBlWzBdCiAgICByZXR1cm4gKDEgLSBnKSAqIGMgKyBnICogKHRvcmNoLmRpYWdvbmFsKGMpLnN1bSgpIC8gZCkgKiB0b3JjaC5leWUoZCwgZHR5cGU9Yy5kdHlwZSkKCgpkZWYgZGl2ZXJnZW5jZXMoc2EsIHNiKToKICAgICIiIkNPUkFMLCBCdXJlcyBhbmQgSmVmZnJleXMgZGl2ZXJnZW5jZXMgb2YgY292YXJpYW5jZSBzYSBmcm9tIHJlZmVyZW5jZSBzYi4iIiIKICAgIGEsIGIgPSBfbm9ybShzYS5kb3VibGUoKSksIF9ub3JtKHNiLmRvdWJsZSgpKQogICAgY29yYWwgPSBmbG9hdCh0b3JjaC5saW5hbGcubm9ybShhIC0gYikgLyB0b3JjaC5saW5hbGcubm9ybShiKS5jbGFtcF9taW4oMWUtMzApKQogICAgZXYsIFUgPSB0b3JjaC5saW5hbGcuZWlnaChiKQogICAgcm9vdCA9IChVICogZXYuY2xhbXBfbWluKDApLnNxcnQoKSkgQCBVLlQKICAgIG0gPSByb290IEAgYSBAIHJvb3QKICAgIGZpZCA9IHRvcmNoLmxpbmFsZy5laWd2YWxzaCgwLjUgKiAobSArIG0uVCkpLmNsYW1wX21pbigwKS5zcXJ0KCkuc3VtKCkKICAgIGJ1cmVzID0gZmxvYXQoKDIuMCAtIDIuMCAqIGZpZCkuY2xhbXBfbWluKDApKQogICAgYTIsIGIyID0gX3NocmluayhhKSwgX3NocmluayhiKQogICAgbGEsIGxiID0gdG9yY2gubGluYWxnLmNob2xlc2t5KGEyKSwgdG9yY2gubGluYWxnLmNob2xlc2t5KGIyKQogICAgdDEgPSB0b3JjaC5jaG9sZXNreV9zb2x2ZShiMiwgbGEpLmRpYWdvbmFsKCkuc3VtKCkgICAgICAjIHRyKEFeLTEgQikKICAgIHQyID0gdG9yY2guY2hvbGVza3lfc29sdmUoYTIsIGxiKS5kaWFnb25hbCgpLnN1bSgpICAgICAgIyB0cihCXi0xIEEpCiAgICBkID0gYS5zaGFwZVswXQogICAgamVmZiA9IGZsb2F0KCh0MSArIHQyKSAvICgyICogZCkgLSAxLjApCiAgICByZXR1cm4geyJjb3JhbCI6IGNvcmFsLCAiYnVyZXMiOiBidXJlcywgImplZmZyZXlzIjogamVmZn0KCgpkZWYgbWFpbigpOgogICAgYXAgPSBhcmdwYXJzZS5Bcmd1bWVudFBhcnNlcigpCiAgICBhcC5hZGRfYXJndW1lbnQoIi0tbW9kZWwiLCBkZWZhdWx0PSJyb2JlcnRhLWJhc2UiKQogICAgYXAuYWRkX2FyZ3VtZW50KCItLXRhc2siLCBkZWZhdWx0PSJjaGVtcHJvdCIpCiAgICBhcC5hZGRfYXJndW1lbnQoIi0tbiIsIHR5cGU9aW50LCBkZWZhdWx0PTEwMjQpCiAgICBhcC5hZGRfYXJndW1lbnQoIi0tYmF0Y2hfc2l6ZSIsIHR5cGU9aW50LCBkZWZhdWx0PTE2KQogICAgYXAuYWRkX2FyZ3VtZW50KCItLWV4dHJhX3JlZnMiLCBhY3Rpb249InN0b3JlX3RydWUiLAogICAgICAgICAgICAgICAgICAgIGhlbHA9ImFsc28gc2NvcmUgbmV3cyB0ZXh0IGFuZCByYW5kb20gdG9rZW5zIGFnYWluc3QgdGhlIHJlZmVyZW5jZSIpCiAgICBhID0gYXAucGFyc2VfYXJncygpCgogICAgZnJvbSB0cmFuc2Zvcm1lcnMgaW1wb3J0IEF1dG9Ub2tlbml6ZXIsIEF1dG9Nb2RlbEZvclNlcXVlbmNlQ2xhc3NpZmljYXRpb24KICAgICMgcnVucyBuZXh0IHRvIHRyYWluaW5nIGpvYnM6IGtlZXAgdGhlIGVpZ2VuZGVjb21wb3NpdGlvbnMgdG8gYSBmZXcgY29yZXMKICAgIHRvcmNoLnNldF9udW1fdGhyZWFkcygyKQogICAgb3MubWFrZWRpcnMoT1VUX0RJUiwgZXhpc3Rfb2s9VHJ1ZSkKICAgIG91dF9wYXRoID0gb3MucGF0aC5qb2luKE9VVF9ESVIsIGYie2EubW9kZWwucmVwbGFjZSgnLycsICdfXycpfV9fe2EudGFza30uanNvbiIpCiAgICBpZiBvcy5wYXRoLmV4aXN0cyhvdXRfcGF0aCk6CiAgICAgICAgcHJpbnQoImV4aXN0czoiLCBvdXRfcGF0aCkKICAgICAgICByZXR1cm4KCiAgICBkZXZpY2UgPSB0b3JjaC5kZXZpY2UoImN1ZGEiIGlmIHRvcmNoLmN1ZGEuaXNfYXZhaWxhYmxlKCkgZWxzZSAiY3B1IikKICAgIHRhc2sgPSBkYXRhX21vZC5sb2FkX3Rhc2soYS50YXNrKQogICAgbWF4X2xlbiA9IGRhdGFfbW9kLlRBU0tfTUFYTEVOLmdldChhLnRhc2ssIDEyOCkKICAgIHRvayA9IEF1dG9Ub2tlbml6ZXIuZnJvbV9wcmV0cmFpbmVkKGEubW9kZWwpCiAgICBtb2RlbCA9IEF1dG9Nb2RlbEZvclNlcXVlbmNlQ2xhc3NpZmljYXRpb24uZnJvbV9wcmV0cmFpbmVkKAogICAgICAgIGEubW9kZWwsIG51bV9sYWJlbHM9dGFza1sibnVtX2xhYmVscyJdKS5mbG9hdCgpLnRvKGRldmljZSkKICAgIGRyaWZ0X21vZC5lbnN1cmVfcGFkZGluZyh0b2ssIG1vZGVsKQogICAgbW9kZWwuZXZhbCgpCiAgICBtb2R1bGVzID0gZHJpZnRfbW9kLmZpbmRfdGFyZ2V0X21vZHVsZXMobW9kZWwpCgogICAgIyB0aGUgcmVmZXJlbmNlIGhhbGYgaXMgZXhhY3RseSB0aGUgcHJvZmlsZXMnIHJlZmVyZW5jZSAoZmlyc3QgbiBwYXNzYWdlcwogICAgIyBhZnRlciB0aGUgZml4ZWQgc2h1ZmZsZSk7IHRoZSBmbG9vciB1c2VzIHRoZSBuZXh0IG4sIGRpc2pvaW50IGZyb20gaXQKICAgIHdpa2kgPSBkYXRhX21vZC5sb2FkX3JlZmVyZW5jZV9jb3JwdXMobl9kb2NzPTIgKiBhLm4pCiAgICBjb3Jwb3JhID0geyJ0YXJnZXQiOiB0YXNrWyJzcGxpdHMiXVsidHJhaW4iXVswXVs6YS5uXSwKICAgICAgICAgICAgICAgInJlZiI6IHdpa2lbOmEubl0sICJyZWYyIjogd2lraVthLm46MiAqIGEubl19CiAgICBpZiBhLmV4dHJhX3JlZnM6CiAgICAgICAgY29ycG9yYVsibmV3cyJdID0gZGF0YV9tb2QubG9hZF9yZWZlcmVuY2VfY29ycHVzKG5fZG9jcz1hLm4sIGtpbmQ9Im5ld3MiKQogICAgICAgIGNvcnBvcmFbInJhbmRvbSJdID0gZGF0YV9tb2QubG9hZF9yZWZlcmVuY2VfY29ycHVzKAogICAgICAgICAgICBuX2RvY3M9YS5uLCBraW5kPSJyYW5kb20iLCB0b2tlbml6ZXI9dG9rLCBuX3Rva2Vucz1tYXhfbGVuIC0gMikKICAgIHQwID0gdGltZS5wZXJmX2NvdW50ZXIoKQogICAgY292ID0ge30KICAgIGZvciBuYW1lLCB0ZXh0cyBpbiBjb3Jwb3JhLml0ZW1zKCk6CiAgICAgICAgY292W25hbWVdID0gZHJpZnRfbW9kLmNvbGxlY3RfY292YXJpYW5jZXMobW9kZWwsIHRvaywgdGV4dHMsIG1vZHVsZXMsIGRldmljZSwKICAgICAgICAgICAgICAgICAgICAgICAgICAgICAgICAgICAgICAgICAgICAgICAgICBtYXhfbGVuPW1heF9sZW4sCiAgICAgICAgICAgICAgICAgICAgICAgICAgICAgICAgICAgICAgICAgICAgICAgICAgYmF0Y2hfc2l6ZT1hLmJhdGNoX3NpemUpCiAgICAgICAgcHJpbnQoZiJ7bmFtZX06IHtsZW4odGV4dHMpfSB0ZXh0cywge3RpbWUucGVyZl9jb3VudGVyKCktdDA6LjBmfXMiLCBmbHVzaD1UcnVlKQogICAgZGVsIG1vZGVsCiAgICBpZiBkZXZpY2UudHlwZSA9PSAiY3VkYSI6CiAgICAgICAgdG9yY2guY3VkYS5lbXB0eV9jYWNoZSgpCgogICAgcGFpcnMgPSBbcCBmb3IgcCBpbiAoInRhcmdldCIsICJyZWYyIiwgIm5ld3MiLCAicmFuZG9tIikgaWYgcCBpbiBjb3ZdCiAgICByZXMgPSB7Im1vZGVsIjogYS5tb2RlbCwgInRhc2siOiBhLnRhc2ssICJuIjogYS5uLCAibWF4X2xlbiI6IG1heF9sZW4sCiAgICAgICAgICAgIm5fdGV4dHMiOiB7azogbGVuKHYpIGZvciBrLCB2IGluIGNvcnBvcmEuaXRlbXMoKX0sCiAgICAgICAgICAgImRpbXMiOiB7bjogbS5pbl9mZWF0dXJlcyBmb3IgbiwgbSBpbiBtb2R1bGVzLml0ZW1zKCl9LCAibW9kdWxlcyI6IHt9fQogICAgZG9uZSA9IHt9CiAgICBmb3IgbmFtZSBpbiBtb2R1bGVzOgogICAgICAgICMgV19RLCBXX0sgYW5kIFdfViByZWFkIG9uZSBpbnB1dCBhbmQgc2hhcmUgb25lIGNvdmFyaWFuY2U6IHNjb3JlIGl0IG9uY2UKICAgICAgICBzaXRlID0gbmFtZS5yZXBsYWNlKCJhdHRlbnRpb24uc2VsZi5rZXkiLCAiYXR0ZW50aW9uLnNlbGYucXVlcnkiKSBcCiAgICAgICAgICAgICAgICAgICAucmVwbGFjZSgiYXR0ZW50aW9uLnNlbGYudmFsdWUiLCAiYXR0ZW50aW9uLnNlbGYucXVlcnkiKQogICAgICAgIGlmIHNpdGUgbm90IGluIGRvbmU6CiAgICAgICAgICAgIHNiID0gY292WyJyZWYiXVtuYW1lXVswXQogICAgICAgICAgICBkb25lW3NpdGVdID0ge3A6IGRpdmVyZ2VuY2VzKGNvdltwXVtuYW1lXVswXSwgc2IpIGZvciBwIGluIHBhaXJzfQogICAgICAgIHJlc1sibW9kdWxlcyJdW25hbWVdID0gZG9uZVtzaXRlXQogICAgcmVzWyJ0X3MiXSA9IHRpbWUucGVyZl9jb3VudGVyKCkgLSB0MAogICAgd2l0aCBvcGVuKG91dF9wYXRoLCAidyIpIGFzIGY6CiAgICAgICAganNvbi5kdW1wKHJlcywgZiwgaW5kZW50PTEpCiAgICBwcmludCgid3JvdGUiLCBvdXRfcGF0aCwgZiIoe3Jlc1sndF9zJ106LjBmfXMpIikKCgppZiBfX25hbWVfXyA9PSAiX19tYWluX18iOgogICAgbWFpbigpCg=="}''')
for name, b64 in PAYLOAD.items():
    with open(os.path.join(WORK, 'src', name), 'wb') as f:
        f.write(base64.b64decode(b64))
print('wrote', len(PAYLOAD), 'modules:', sorted(PAYLOAD))


## Fetch datasets

In [ ]:
subprocess.run([sys.executable, 'src/download_data.py'], cwd=WORK, check=True)


## Restore results from earlier sessions
Kaggle: any attached `drift_results.zip` (a previous version's output) is unpacked. Colab: `runs/results` is linked to Google Drive, so earlier results are already there.

In [ ]:
import glob, zipfile, shutil
os.makedirs('runs/profiles', exist_ok=True)
if ON_COLAB:
    from google.colab import drive
    drive.mount('/content/drive')
    DRIVE = '/content/drive/MyDrive/drift_results'
    os.makedirs(os.path.join(DRIVE, 'results'), exist_ok=True)
    if os.path.isdir('runs/results') and not os.path.islink('runs/results'):
        for p in glob.glob('runs/results/*.json'):
            shutil.copy(p, os.path.join(DRIVE, 'results'))
        shutil.rmtree('runs/results')
    if not os.path.exists('runs/results'):
        os.symlink(os.path.join(DRIVE, 'results'), 'runs/results')
os.makedirs('runs/results', exist_ok=True)
restored = 0
for z in sorted(glob.glob('/kaggle/input/**/drift_results.zip', recursive=True)):
    with zipfile.ZipFile(z) as zf:
        for n in zf.namelist():
            if n.startswith('results/') and n.endswith('.json'):
                dst = os.path.join('runs', n)
                if not os.path.exists(dst):
                    with zf.open(n) as s, open(dst, 'wb') as d:
                        d.write(s.read())
                    restored += 1
    print('restored from', z)
# a zip uploaded as a Kaggle Dataset arrives already unpacked
for p in glob.glob('/kaggle/input/**/results/*.json', recursive=True):
    dst = os.path.join('runs/results', os.path.basename(p))
    if not os.path.exists(dst):
        shutil.copy(p, dst)
        restored += 1
print('restored', restored, 'new results;',
      len(glob.glob('runs/results/*.json')), 'results present in total')


## Configure

`DRIFT_AMP=1` turns on fp16 autocast, a large speed-up on T4/P100, applied identically to every method so comparisons stay matched. Runs are split across all visible GPUs (two on Kaggle's T4 x2). No new run starts after `DEADLINE_HOURS`; the margin leaves room for the last run to finish.

In [ ]:
os.environ['DRIFT_AMP'] = '1'
# fail a stalled checkpoint download instead of hanging on it
os.environ['HF_HUB_DOWNLOAD_TIMEOUT'] = '60'
os.environ['HF_HUB_ETAG_TIMEOUT'] = '60'
MODEL = 'roberta-base'
DECODER = 'HuggingFaceTB/SmolLM2-360M'   # decoder SLM for the generality check
DEADLINE_HOURS = 1.3          # Kaggle kills a session at 12 h
DEADLINE = SESSION_START + DEADLINE_HOURS * 3600
NEED_PROFILES = True
os.makedirs('logs', exist_ok=True)

def snapshot(label=''):
    """Zip the result JSONs and profile summaries (not the large .pt
    profile tensors, which are recomputed cheaply)."""
    paths = sorted(glob.glob('runs/results/*.json')) + \
            sorted(glob.glob('runs/profiles/*.json')) + \
            sorted(glob.glob('runs/divergence/*.json'))
    out = os.path.join(WORK, 'drift_results.zip')
    with zipfile.ZipFile(out, 'w', zipfile.ZIP_DEFLATED) as z:
        for p in paths:
            z.write(p, os.path.relpath(p, 'runs'))
    if ON_COLAB:
        shutil.copy(out, DRIVE)
    nres = sum(1 for p in paths if 'results' in p)
    print(f'[snapshot {label}] {nres} results -> {out}', flush=True)

def run_plan(plan, seeds='1,2,3', model=MODEL):
    """Run one experiment plan, one worker per GPU, then snapshot."""
    if time.time() > DEADLINE:
        print(f'[{plan}] skipped: session deadline reached. Start a new '
              'session to continue.')
        return
    base = [sys.executable, 'src/grid.py', '--plan', plan,
            '--model', model, '--seeds', seeds]
    if NEED_PROFILES:
        subprocess.run(base + ['--profiles_only', '--deadline', str(DEADLINE)],
                       check=False)
    procs, logs = [], []
    for g in range(NGPU):
        log = f'logs/{plan}_gpu{g}.log'
        env = dict(os.environ, CUDA_VISIBLE_DEVICES=str(g))
        procs.append(subprocess.Popen(
            base + ['--shard', f'{g}/{NGPU}', '--deadline', str(DEADLINE),
                    '--no_profiles'],
            env=env, stdout=open(log, 'w'), stderr=subprocess.STDOUT))
        logs.append(log)
    t0 = last_snap = last_print = time.time()
    while any(p.poll() is None for p in procs):
        # poll often, so a plan with nothing left to do costs seconds, not minutes
        time.sleep(5)
        if time.time() - last_print < 120:
            continue
        last_print = time.time()
        done = sum(open(l, errors='ignore').read().count('\nDONE ') for l in logs)
        print(f'[{plan}] {(time.time()-t0)/60:.0f} min, {done} runs finished '
              f'this session', flush=True)
        # a session stopped from outside (quota, time limit) keeps what the
        # last snapshot holds, so snapshot during long plans too
        if time.time() - last_snap > 900:
            snapshot(plan + ' (partial)')
            last_snap = time.time()
    for l in logs:
        txt = open(l, errors='ignore').read()
        print(f'--- tail of {l} ---')
        print(txt[-1500:])
        if 'Traceback' in txt:
            print(f'!! errors in {l}: search it for Traceback')
    snapshot(plan)


## Smoke test of the post-audit code paths
Linear head, rsLoRA scale, stored predictions (single- and multi-label), the decoder on 512-token HoC documents at batch 8 (memory and speed), and the divergence script on a small sample.

In [ ]:
import json, glob
def sh(*a, timeout=3600):
    r = subprocess.run([sys.executable, *a], capture_output=True, text=True,
                       timeout=timeout)
    tail = (r.stdout + r.stderr).strip().splitlines()[-6:]
    print('$', ' '.join(a[:14]), '->', 'OK' if r.returncode == 0 else
          f'EXIT {r.returncode}', flush=True)
    print('   ' + '\n   '.join(tail), flush=True)
    return r.returncode
fails = 0
enc = ['src/run.py', '--model', MODEL, '--epochs', '1', '--seed', '1', '--amp',
       '--tag', 'rsmoke2']
for extra in (['--task', 'chemprot', '--method', 'lora', '--head', 'linear',
               '--max_train', '600'],
              ['--task', 'chemprot', '--method', 'bitfit', '--head', 'linear',
               '--max_train', '600', '--lr', '1e-3'],
              ['--task', 'chemprot', '--method', 'lora', '--scaling', 'rslora',
               '--budget_rank', '2', '--max_train', '600'],
              ['--task', 'hoc', '--method', 'lora', '--max_train', '200',
               '--batch_size', '16', '--max_len', '512']):
    t1 = time.time(); fails += sh(*enc, *extra) != 0
    print(f'   {time.time()-t1:.0f}s', flush=True)
t1 = time.time()
fails += sh('src/profile_drift.py', '--model', DECODER, '--task', 'hoc', '--n_ref', '64',
            '--n_dom', '64', '--taus', '0.0,0.95') != 0
print(f'   decoder HoC profile {time.time()-t1:.0f}s', flush=True)
for m in ('lora', 'eva'):
    t1 = time.time()
    fails += sh('src/run.py', '--model', DECODER, '--task', 'hoc', '--method', m,
                '--epochs', '1', '--max_train', '160', '--batch_size', '8',
                '--max_len', '512', '--seed', '1', '--amp', '--lr', '3e-4',
                '--n_ref', '64', '--n_dom', '64', '--tag', 'rsmoke2') != 0
    print(f'   decoder HoC {m}: {time.time()-t1:.0f}s', flush=True)
t1 = time.time()
fails += sh('src/divergence.py', '--task', 'chemprot', '--n', '64', '--extra_refs') != 0
print(f'   divergence (n=64) {time.time()-t1:.0f}s', flush=True)
for p in sorted(glob.glob('runs/results/*rsmoke2*.json')):
    r = json.load(open(p)); a = r['args']; res = r['result']; m = r['metric']
    tp, tg = res.get('test_preds'), res.get('test_gold')
    print(a['model'].split('/')[-1], a['task'], a['method'], 'head', a.get('head'),
          'scaling', a.get('scaling'), '| test', round(res['test'][m], 4),
          '| adapter', res['params_adapter'], 'head params', res['params_head'],
          '| preds', None if tp is None else len(tp), 'gold', None if tg is None else len(tg),
          '| peak GB', round(res['peak_mem_bytes'] / 1e9, 2), '| train_s', round(res['train_time_s']),
          '| steps', res['steps'])
    fails += tp is None or tg is None or len(tp) != len(tg)
for p in glob.glob('runs/divergence/*.json'):
    d = json.load(open(p))
    first = next(iter(d['modules'].values()))
    print(os.path.basename(p), 'pairs', list(first), 'first module', first)
print('SMOKE2', 'PASSED' if fails == 0 else f'FAILED ({fails})')
snapshot('revision_smoke2')


## Collect results
Download **drift_results.zip** (Kaggle: Output panel; Colab: `MyDrive/drift_results/`).

In [ ]:
snapshot('final')
subprocess.run([sys.executable, 'src/analyze.py', '--model', MODEL, '--dump'],
               check=False)
